In [3]:
from estnltk import Text
from estnltk.taggers import VabamorfTagger, VabamorfAnalyzer
from estnltk_neural.taggers import StanzaSyntaxTagger
from estnltk.converters import text_to_json
import sys, os
import re
import csv
import pandas as pd
import json
from tqdm import tqdm

In [4]:
pd.set_option('display.max_colwidth', None)
pd.set_option("display.show_dimensions", True)

In [11]:
RESULT_DIR = "../results/"

DATA_FILE = RESULT_DIR+ "n80_examples_large_v01/gpt_v02/" + "gpt_10K_b10_run01.csv"

In [12]:
df1 = pd.read_csv(DATA_FILE, encoding="utf-8", sep=",")

In [7]:
morf_tagger = VabamorfAnalyzer(output_layer='morph_analysis')
stanza_tagger = StanzaSyntaxTagger(input_type='morph_extended', input_morph_layer='morph_extended')


In [7]:
#morf_tagger.analyze_token

In [8]:
morf_tagger.analyze_token("kaitseliitlasest")

[{'root': 'kaitse_liitlane',
  'root_tokens': ['kaitse', 'liitlane'],
  'ending': 'st',
  'clitic': '',
  'partofspeech': 'S',
  'form': 'sg el',
  'lemma': 'kaitseliitlane'}]

In [8]:
def get_analysis(ex):
    txt = Text(ex.iloc[0]["sentence"])
    txt.tag_layer("words")
    txt.tag_layer("sentences")
    morf_tagger.tag(txt)
    txt.tag_layer('morph_extended')
    stanza_tagger.tag( txt )
    
    head_loc = int(ex.iloc[0]["head_loc"])-1
    head_analysis = txt.morph_analysis[head_loc]
    
    roots = list(head_analysis.root)
    forms = list(head_analysis.form)
    poss = list(head_analysis.partofspeech)

    variants = []
    for r, f, pos in zip(roots, forms, poss):
        variants.append((r, f, pos))

    return txt, txt.morph_analysis, head_analysis, variants

In [45]:
df1[df1["sentence"].str.contains("joobes mees Maardus")]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,classification,explanation,classification2,explanation2,timex_tag,ekilex_tag,ner_tag
7454,2263222,3623029,6,magama,NaN,in,joove,joobes,25.jaanuaril kell 22.20 magas joobes mees Maardus Keemikute tänaval .,no,"The phrase 'joobes' was classified as 'no' because it describes a state of intoxication, not a location.",no,"It was classified as not adverbial of place because 'joobes' describes the state or condition of the man (drunken), not a location.",NaN,state,NaN


In [13]:


ex = df1[df1["sentence"].str.contains("Kohtumine kulges tavapäraselt hallilt")]
text, analysis, word_analysis, head_variants = get_analysis(ex)
print(text, "\n\nhead_word:", ex.iloc[0].form)
analysis

Text(text='Kohtumine kulges tavapäraselt hallilt .') 

head_word: hallilt


Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('Kohtumine', [{'normalized_text': 'Kohtumine', 'lemma': 'Kohtumine', 'root': 'Kohtumine', 'root_tokens': ['Kohtumine'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'H'}, {'normalized_text': 'Kohtumine', 'lemma': 'kohtumine', 'root': 'kohtumine', 'root_tokens': ['kohtumine'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('kulges', [{'normalized_text': 'kulges', 'lemma': 'kulgema', 'root': 'kulge', 'root_tokens': ['kulge'], 'ending': 's', 'clitic': '', 'form': 's', 'partofspeech': 'V'}]),
Span('tavapäraselt', [{'normalized_text': 'tavapäraselt', 'lemma': 'tavapärane', 'root': 'tava_pärane', 'root_tokens': ['tava', 'pärane'], 'ending': 'lt', 'clitic': '', 'form': 'sg abl', 'partofspeech': 'A'}, {'normalized_text': 'tavapäraselt', 'lemma': 'tavapäraselt', 'root': 'tava_päraselt', 'root_tokens': ['tava', 'päraselt'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}]),
Span('hallilt', [{'normalized_text': 'hallilt', 'lemma': 'hall', 'root': 'hall', 'root_tokens': ['hall'], 'ending': 'lt', 'clitic': '', 'form': 'sg abl', 'partofspeech': 'A'}, {'normalized_text': 'hallilt', 'lemma': 'hall', 'root': 'hall', 'root_tokens': ['hall'], 'ending': 'lt', 'clitic': '', 'form': 'sg abl', 'partofspeech': 'S'}, {'normalized_text': 'hallilt', 'lemma': 'hallilt', 'root': 'hallilt', 'root_tokens': ['hallilt'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}]),
Span('.', [{'normalized_text': '.', 'lemma': '.', 'root': '.', 'root_tokens': ['.'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}])])

In [14]:
text.stanza_syntax

Layer(name='stanza_syntax', attributes=('id', 'lemma', 'upostag', 'xpostag', 'feats', 'head', 'deprel', 'deps', 'misc'), spans=SL[Span('Kohtumine', [{'id': 1, 'lemma': 'Kohtumine', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('prop', 'prop'), ('sg', 'sg'), ('nom', 'nom')]), 'head': 2, 'deprel': 'nsubj', 'deps': '_', 'misc': '_'}]),
Span('kulges', [{'id': 2, 'lemma': 'kulgema', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('mod', 'mod'), ('indic', 'indic'), ('impf', 'impf'), ('ps3', 'ps3'), ('sg', 'sg'), ('ps', 'ps'), ('af', 'af')]), 'head': 0, 'deprel': 'root', 'deps': '_', 'misc': '_'}]),
Span('tavapäraselt', [{'id': 3, 'lemma': 'tavapärane', 'upostag': 'A', 'xpostag': 'A', 'feats': OrderedDict([('pos', 'pos'), ('sg', 'sg'), ('abl', 'abl')]), 'head': 4, 'deprel': 'amod', 'deps': '_', 'misc': '_'}]),
Span('hallilt', [{'id': 4, 'lemma': 'hall', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('abl', 'abl')]), 'head': 2, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('.', [{'id': 5, 'lemma': '.', 'upostag': 'Z', 'xpostag': 'Z', 'feats': OrderedDict(), 'head': 2, 'deprel': 'punct', 'deps': '_', 'misc': '_'}])])

In [39]:

ex = df1[df1["sentence"].str.contains("STV pakutavad kanalid jõuavad 25 000-sse koju .")]
text, analysis, word_analysis, head_variants = get_analysis(ex)
print(text, "\n\nhead_word:", ex.iloc[0].form)
analysis

Text(text='STV pakutavad kanalid jõuavad 25 000-sse koju .') 

head_word: 25 000-sse


Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('STV', [{'normalized_text': 'STV', 'lemma': 'STV', 'root': 'STV', 'root_tokens': ['STV'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'Y'}]),
Span('pakutavad', [{'normalized_text': 'pakutavad', 'lemma': 'pakutama', 'root': 'pakuta', 'root_tokens': ['pakuta'], 'ending': 'vad', 'clitic': '', 'form': 'vad', 'partofspeech': 'V'}, {'normalized_text': 'pakutavad', 'lemma': 'pakutav', 'root': 'pakutav', 'root_tokens': ['pakutav'], 'ending': 'd', 'clitic': '', 'form': 'pl n', 'partofspeech': 'A'}]),
Span('kanalid', [{'normalized_text': 'kanalid', 'lemma': 'kanal', 'root': 'kanal', 'root_tokens': ['kanal'], 'ending': 'd', 'clitic': '', 'form': 'pl n', 'partofspeech': 'S'}]),
Span('jõuavad', [{'normalized_text': 'jõuavad', 'lemma': 'jõudma', 'root': 'jõud', 'root_tokens': ['jõud'], 'ending': 'vad', 'clitic': '', 'form': 'vad', 'partofspeech': 'V'}]),
Span('25 000-sse', [{'normalized_text': '25000-sse', 'lemma': '25000', 'root': '25000', 'root_tokens': ['25000'], 'ending': 'sse', 'clitic': '', 'form': 'sg ill', 'partofspeech': 'N'}]),
Span('koju', [{'normalized_text': 'koju', 'lemma': 'kodu', 'root': 'kodu', 'root_tokens': ['kodu'], 'ending': '0', 'clitic': '', 'form': 'adt', 'partofspeech': 'S'}]),
Span('.', [{'normalized_text': '.', 'lemma': '.', 'root': '.', 'root_tokens': ['.'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}])])

In [27]:
ex = df1[df1["sentence"].str.contains("kõrguselt jõkke")]
text, analysis, word_analysis, head_variants = get_analysis(ex)
print(text, "\n\nhead_word:", ex.iloc[0].form)
head_variants

Text(text='7. juulil sõitis Koluvere sillalt nelja meetri kõrguselt jõkke 30 aastat vana Volvo-buss .') 

head_word: kõrguselt


[('kõrgune', 'sg abl', 'A'),
 ('kõrgus', 'sg abl', 'S'),
 ('kõrguse=lt', '', 'D')]

In [29]:
text.stanza_syntax

Layer(name='stanza_syntax', attributes=('id', 'lemma', 'upostag', 'xpostag', 'feats', 'head', 'deprel', 'deps', 'misc'), spans=SL[Span('7.', [{'id': 1, 'lemma': '7.', 'upostag': 'N', 'xpostag': 'N', 'feats': OrderedDict([('ord', 'ord'), ('<?>', '<?>'), ('roman', 'roman')]), 'head': 2, 'deprel': 'amod', 'deps': '_', 'misc': '_'}]),
Span('juulil', [{'id': 2, 'lemma': 'juuli', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('ad', 'ad')]), 'head': 3, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('sõitis', [{'id': 3, 'lemma': 'sõitma', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('aux', 'aux'), ('indic', 'indic'), ('impf', 'impf'), ('ps3', 'ps3'), ('sg', 'sg'), ('ps', 'ps'), ('af', 'af')]), 'head': 0, 'deprel': 'root', 'deps': '_', 'misc': '_'}]),
Span('Koluvere', [{'id': 4, 'lemma': 'Koluvere', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('prop', 'prop'), ('sg', 'sg'), ('gen', 'gen')]), 'head': 5, 'deprel': 'nmod', 'deps': '_', 'misc': '_'}]),
Span('sillalt', [{'id': 5, 'lemma': 'sild', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('abl', 'abl')]), 'head': 3, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('nelja', [{'id': 6, 'lemma': 'neli', 'upostag': 'N', 'xpostag': 'N', 'feats': OrderedDict([('card', 'card'), ('sg', 'sg'), ('part', 'part'), ('l', 'l')]), 'head': 7, 'deprel': 'nummod', 'deps': '_', 'misc': '_'}]),
Span('meetri', [{'id': 7, 'lemma': 'meeter', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('gen', 'gen')]), 'head': 8, 'deprel': 'nmod', 'deps': '_', 'misc': '_'}]),
Span('kõrguselt', [{'id': 8, 'lemma': 'kõrgus', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('abl', 'abl')]), 'head': 3, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('jõkke', [{'id': 9, 'lemma': 'jõgi', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('adit', 'adit')]), 'head': 3, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('30', [{'id': 10, 'lemma': '30', 'upostag': 'N', 'xpostag': 'N', 'feats': OrderedDict([('card', 'card'), ('<?>', '<?>'), ('digit', 'digit')]), 'head': 11, 'deprel': 'nummod', 'deps': '_', 'misc': '_'}]),
Span('aastat', [{'id': 11, 'lemma': 'aasta', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('part', 'part')]), 'head': 3, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('vana', [{'id': 12, 'lemma': 'vana', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('part', 'part')]), 'head': 13, 'deprel': 'nmod', 'deps': '_', 'misc': '_'}]),
Span('Volvo-buss', [{'id': 13, 'lemma': 'Volvo-buss', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('nom', 'nom')]), 'head': 3, 'deprel': 'nsubj', 'deps': '_', 'misc': '_'}]),
Span('.', [{'id': 14, 'lemma': '.', 'upostag': 'Z', 'xpostag': 'Z', 'feats': OrderedDict(), 'head': 3, 'deprel': 'punct', 'deps': '_', 'misc': '_'}])])

In [12]:
ex = df1[df1["sentence"].str.contains("Disney rahakirstudesse")]
text, analysis, word_analysis, head_variants = get_analysis(ex)
print(text, "\n\nhead_word:", ex.iloc[0].form)
head_variants


Text(text='Mitmele teisele stuudiole on ühtäkki tulnud mõte , et ka nemad võiksid saada osa neist tohututest summadest , mis seni tänu joonisfilmidele on jooksnud Disney rahakirstudesse .') 

head_word: joonisfilmidele


[('joonis_film', 'pl all', 'S')]

In [14]:
text.stanza_syntax

Layer(name='stanza_syntax', attributes=('id', 'lemma', 'upostag', 'xpostag', 'feats', 'head', 'deprel', 'deps', 'misc'), spans=SL[Span('Mitmele', [{'id': 1, 'lemma': 'mitu', 'upostag': 'P', 'xpostag': 'P', 'feats': OrderedDict([('sg', 'sg'), ('all', 'all')]), 'head': 3, 'deprel': 'det', 'deps': '_', 'misc': '_'}]),
Span('teisele', [{'id': 2, 'lemma': 'teine', 'upostag': 'N', 'xpostag': 'N', 'feats': OrderedDict([('ord', 'ord'), ('sg', 'sg'), ('all', 'all'), ('roman', 'roman')]), 'head': 3, 'deprel': 'det', 'deps': '_', 'misc': '_'}]),
Span('stuudiole', [{'id': 3, 'lemma': 'stuudio', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('all', 'all')]), 'head': 6, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('on', [{'id': 4, 'lemma': 'olema', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('aux', 'aux'), ('indic', 'indic'), ('pres', 'pres'), ('ps3', 'ps3'), ('pl', 'pl'), ('ps', 'ps'), ('af', 'af')]), 'head': 6, 'deprel': 'aux', 'deps': '_', 'misc': '_'}]),
Span('ühtäkki', [{'id': 5, 'lemma': 'ühtäkki', 'upostag': 'D', 'xpostag': 'D', 'feats': OrderedDict(), 'head': 6, 'deprel': 'advmod', 'deps': '_', 'misc': '_'}]),
Span('tulnud', [{'id': 6, 'lemma': 'tulema', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('main', 'main'), ('indic', 'indic'), ('impf', 'impf'), ('ps', 'ps'), ('neg', 'neg')]), 'head': 0, 'deprel': 'root', 'deps': '_', 'misc': '_'}]),
Span('mõte', [{'id': 7, 'lemma': 'mõte', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('nom', 'nom')]), 'head': 6, 'deprel': 'nsubj', 'deps': '_', 'misc': '_'}]),
Span(',', [{'id': 8, 'lemma': ',', 'upostag': 'Z', 'xpostag': 'Z', 'feats': OrderedDict(), 'head': 13, 'deprel': 'punct', 'deps': '_', 'misc': '_'}]),
Span('et', [{'id': 9, 'lemma': 'et', 'upostag': 'J', 'xpostag': 'J', 'feats': OrderedDict([('sub', 'sub'), ('crd', 'crd')]), 'head': 13, 'deprel': 'mark', 'deps': '_', 'misc': '_'}]),
Span('ka', [{'id': 10, 'lemma': 'ka', 'upostag': 'D', 'xpostag': 'D', 'feats': OrderedDict(), 'head': 11, 'deprel': 'advmod', 'deps': '_', 'misc': '_'}]),
Span('nemad', [{'id': 11, 'lemma': 'tema', 'upostag': 'P', 'xpostag': 'P', 'feats': OrderedDict([('pl', 'pl'), ('nom', 'nom')]), 'head': 13, 'deprel': 'nsubj', 'deps': '_', 'misc': '_'}]),
Span('võiksid', [{'id': 12, 'lemma': 'võima', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('mod', 'mod'), ('cond', 'cond'), ('pres', 'pres'), ('ps2', 'ps2'), ('sg', 'sg'), ('ps', 'ps'), ('af', 'af')]), 'head': 13, 'deprel': 'aux', 'deps': '_', 'misc': '_'}]),
Span('saada', [{'id': 13, 'lemma': 'saatma', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('mod', 'mod'), ('indic', 'indic'), ('pres', 'pres'), ('ps', 'ps'), ('neg', 'neg')]), 'head': 7, 'deprel': 'acl', 'deps': '_', 'misc': '_'}]),
Span('osa', [{'id': 14, 'lemma': 'osa', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('gen', 'gen')]), 'head': 13, 'deprel': 'obj', 'deps': '_', 'misc': '_'}]),
Span('neist', [{'id': 15, 'lemma': 'see', 'upostag': 'P', 'xpostag': 'P', 'feats': OrderedDict([('pl', 'pl'), ('el', 'el')]), 'head': 17, 'deprel': 'det', 'deps': '_', 'misc': '_'}]),
Span('tohututest', [{'id': 16, 'lemma': 'tohutu', 'upostag': 'A', 'xpostag': 'A', 'feats': OrderedDict([('pos', 'pos'), ('pl', 'pl'), ('el', 'el')]), 'head': 17, 'deprel': 'amod', 'deps': '_', 'misc': '_'}]),
Span('summadest', [{'id': 17, 'lemma': 'summa', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('pl', 'pl'), ('el', 'el')]), 'head': 14, 'deprel': 'nmod', 'deps': '_', 'misc': '_'}]),
Span(',', [{'id': 18, 'lemma': ',', 'upostag': 'Z', 'xpostag': 'Z', 'feats': OrderedDict(), 'head': 24, 'deprel': 'punct', 'deps': '_', 'misc': '_'}]),
Span('mis', [{'id': 19, 'lemma': 'mis', 'upostag': 'P', 'xpostag': 'P', 'feats': OrderedDict([('sg', 'sg'), ('nom', 'nom')]), 'head': 24, 'deprel': 'nsubj', 'deps': '_', 'misc': '_'}]),
Span('seni', [{'id': 20, 'lemma': 'seni', 'upos

In [17]:

ex = df1[df1["sentence"].str.contains("inimesi lamas igal pool")]
text, analysis, word_analysis, head_variants = get_analysis(ex)
print(text, "\n\nhead_word:", ex.iloc[0].form)
analysis

Text(text='Üks pealtnägija rääkis BBCle , et nägi , kuidas kahekordne buss oli lõhki nagu " karp sardiine " ning inimesi lamas igal pool .') 

head_word: igal


Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('Üks', [{'normalized_text': 'Üks', 'lemma': 'üks', 'root': 'üks', 'root_tokens': ['üks'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'N'}, {'normalized_text': 'Üks', 'lemma': 'üks', 'root': 'üks', 'root_tokens': ['üks'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'P'}]),
Span('pealtnägija', [{'normalized_text': 'pealtnägija', 'lemma': 'pealtnägija', 'root': 'pealt_nägija', 'root_tokens': ['pealt', 'nägija'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'S'}, {'normalized_text': 'pealtnägija', 'lemma': 'pealtnägija', 'root': 'pealt_nägija', 'root_tokens': ['pealt', 'nägija'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('rääkis', [{'normalized_text': 'rääkis', 'lemma': 'rääkima', 'root': 'rääki', 'root_tokens': ['rääki'], 'ending': 's', 'clitic': '', 'form': 's', 'partofspeech': 'V'}]),
Span('BBCle', [{'normalized_text': 'BBCle', 'lemma': 'BBC', 'root': 'BBC', 'root_tokens': ['BBC'], 'ending': 'le', 'clitic': '', 'form': 'sg all', 'partofspeech': 'Y'}]),
Span(',', [{'normalized_text': ',', 'lemma': ',', 'root': ',', 'root_tokens': [','], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('et', [{'normalized_text': 'et', 'lemma': 'et', 'root': 'et', 'root_tokens': ['et'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'J'}]),
Span('nägi', [{'normalized_text': 'nägi', 'lemma': 'nägema', 'root': 'näge', 'root_tokens': ['näge'], 'ending': 'i', 'clitic': '', 'form': 's', 'partofspeech': 'V'}]),
Span(',', [{'normalized_text': ',', 'lemma': ',', 'root': ',', 'root_tokens': [','], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('kuidas', [{'normalized_text': 'kuidas', 'lemma': 'kuidas', 'root': 'kuidas', 'root_tokens': ['kuidas'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}]),
Span('kahekordne', [{'normalized_text': 'kahekordne', 'lemma': 'kahekordne', 'root': 'kahe_kordne', 'root_tokens': ['kahe', 'kordne'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'A'}]),
Span('buss', [{'normalized_text': 'buss', 'lemma': 'buss', 'root': 'buss', 'root_tokens': ['buss'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('oli', [{'normalized_text': 'oli', 'lemma': 'olema', 'root': 'ole', 'root_tokens': ['ole'], 'ending': 'i', 'clitic': '', 'form': 's', 'partofspeech': 'V'}]),
Span('lõhki', [{'normalized_text': 'lõhki', 'lemma': 'lõhki', 'root': 'lõhki', 'root_tokens': ['lõhki'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}]),
Span('nagu', [{'normalized_text': 'nagu', 'lemma': 'nagu', 'root': 'nagu', 'root_tokens': ['nagu'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}, {'normalized_text': 'nagu', 'lemma': 'nagu', 'root': 'nagu', 'root_tokens': ['nagu'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'J'}]),
Span('"', [{'normalized_text': '"', 'lemma': '"', 'root': '"', 'root_tokens': ['"'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('karp', [{'normalized_text': 'karp', 'lemma': 'karp', 'root': 'karp', 'root_tokens': ['karp'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('sardiine', [{'normalized_text': 'sardiine', 'lemma': 'sardiin', 'root': 'sardiin', 'root_tokens': ['sardiin'], 'ending': 'e', 'clitic': '', 'form': 'pl p', 'partofspeech': 'S'}]),
Span('"', [{'normalized_text': '"', 'lemma': '"', 'root': '"', 'root_tokens': ['"'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('ning', [{'normalized_text': 'ning', 'lemma': 'ning', 'root': 'ning', 'root_tokens': ['ning'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'J'}]),
Span('inimesi', [{'normalized_text': 'inimesi', 'lemma': 'inimene', 'root': 'inimene', 'root_tokens': ['inimene'], 'ending': 'i', 'clitic': '', 'form': 'pl p', 'parto

In [18]:
text.stanza_syntax

Layer(name='stanza_syntax', attributes=('id', 'lemma', 'upostag', 'xpostag', 'feats', 'head', 'deprel', 'deps', 'misc'), spans=SL[Span('Üks', [{'id': 1, 'lemma': 'üks', 'upostag': 'N', 'xpostag': 'N', 'feats': OrderedDict([('card', 'card'), ('sg', 'sg'), ('nom', 'nom'), ('l', 'l')]), 'head': 2, 'deprel': 'det', 'deps': '_', 'misc': '_'}]),
Span('pealtnägija', [{'id': 2, 'lemma': 'pealtnägija', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('gen', 'gen')]), 'head': 3, 'deprel': 'nsubj', 'deps': '_', 'misc': '_'}]),
Span('rääkis', [{'id': 3, 'lemma': 'rääkima', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('mod', 'mod'), ('indic', 'indic'), ('impf', 'impf'), ('ps3', 'ps3'), ('sg', 'sg'), ('ps', 'ps'), ('af', 'af')]), 'head': 0, 'deprel': 'root', 'deps': '_', 'misc': '_'}]),
Span('BBCle', [{'id': 4, 'lemma': 'BBC', 'upostag': 'Y', 'xpostag': 'Y', 'feats': OrderedDict([('nominal', 'nominal'), ('sg', 'sg'), ('all', 'all')]), 'head': 3, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span(',', [{'id': 5, 'lemma': ',', 'upostag': 'Z', 'xpostag': 'Z', 'feats': OrderedDict(), 'head': 7, 'deprel': 'punct', 'deps': '_', 'misc': '_'}]),
Span('et', [{'id': 6, 'lemma': 'et', 'upostag': 'J', 'xpostag': 'J', 'feats': OrderedDict([('sub', 'sub'), ('crd', 'crd')]), 'head': 7, 'deprel': 'mark', 'deps': '_', 'misc': '_'}]),
Span('nägi', [{'id': 7, 'lemma': 'nägema', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('aux', 'aux'), ('indic', 'indic'), ('impf', 'impf'), ('ps3', 'ps3'), ('sg', 'sg'), ('ps', 'ps'), ('af', 'af')]), 'head': 3, 'deprel': 'ccomp', 'deps': '_', 'misc': '_'}]),
Span(',', [{'id': 8, 'lemma': ',', 'upostag': 'Z', 'xpostag': 'Z', 'feats': OrderedDict(), 'head': 13, 'deprel': 'punct', 'deps': '_', 'misc': '_'}]),
Span('kuidas', [{'id': 9, 'lemma': 'kuidas', 'upostag': 'D', 'xpostag': 'D', 'feats': OrderedDict(), 'head': 13, 'deprel': 'mark', 'deps': '_', 'misc': '_'}]),
Span('kahekordne', [{'id': 10, 'lemma': 'kahekordne', 'upostag': 'A', 'xpostag': 'A', 'feats': OrderedDict([('pos', 'pos'), ('sg', 'sg'), ('nom', 'nom')]), 'head': 11, 'deprel': 'amod', 'deps': '_', 'misc': '_'}]),
Span('buss', [{'id': 11, 'lemma': 'buss', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('nom', 'nom')]), 'head': 13, 'deprel': 'nsubj:cop', 'deps': '_', 'misc': '_'}]),
Span('oli', [{'id': 12, 'lemma': 'olema', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('mod', 'mod'), ('indic', 'indic'), ('impf', 'impf'), ('ps3', 'ps3'), ('sg', 'sg'), ('ps', 'ps'), ('af', 'af')]), 'head': 13, 'deprel': 'cop', 'deps': '_', 'misc': '_'}]),
Span('lõhki', [{'id': 13, 'lemma': 'lõhki', 'upostag': 'D', 'xpostag': 'D', 'feats': OrderedDict(), 'head': 7, 'deprel': 'ccomp', 'deps': '_', 'misc': '_'}]),
Span('nagu', [{'id': 14, 'lemma': 'nagu', 'upostag': 'J', 'xpostag': 'J', 'feats': OrderedDict([('sub', 'sub'), ('crd', 'crd')]), 'head': 17, 'deprel': 'mark', 'deps': '_', 'misc': '_'}]),
Span('"', [{'id': 15, 'lemma': '"', 'upostag': 'Z', 'xpostag': 'Z', 'feats': OrderedDict(), 'head': 17, 'deprel': 'punct', 'deps': '_', 'misc': '_'}]),
Span('karp', [{'id': 16, 'lemma': 'karp', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('nom', 'nom')]), 'head': 17, 'deprel': 'nmod', 'deps': '_', 'misc': '_'}]),
Span('sardiine', [{'id': 17, 'lemma': 'sardiin', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('pl', 'pl'), ('part', 'part')]), 'head': 13, 'deprel': 'advcl', 'deps': '_', 'misc': '_'}]),
Span('"', [{'id': 18, 'lemma': '"', 'upostag': 'Z', 'xpostag': 'Z', 'feats': OrderedDict(), 'head': 17, 'deprel': 'punct', 'deps': '_', 'misc': '_'}]),
Span('ning', [{'id': 19, 'lemma': 'ning', 'upostag': 'J', 'xpostag': 'J', 'feats': OrderedDict([('sub', 'sub'), ('crd', 'crd')]), 'head': 21, 'deprel': 'cc', 'deps': '_', 'misc': '_'}]),
Span('inimesi', [{'id': 20, 'lemma': 'inimene', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([

In [9]:
ex = df1[df1["sentence"].str.contains("kaitseliitlasest mehelt")]
text, analysis, word_analysis, head_variants = get_analysis(ex)
print(text, "\n\nhead_word:", ex.iloc[0].form)
head_variants

Text(text='Poolesajal kaitseliitlasel , politseinikul ja päästeametnikul ei õnnestunud läinud nädala lõpuks leida Jõhvi lähedastest metsadest 23-aastast neiut , kes varastas kasuõe kaitseliitlasest mehelt püstoli ja lubas end tappa .') 

head_word: kaitseliitlasest


[('kaitse_liitlane', 'sg el', 'S')]

In [10]:
analysis

Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('Poolesajal', [{'normalized_text': 'Poolesajal', 'lemma': 'poolsada', 'root': 'pool_sada', 'root_tokens': ['pool', 'sada'], 'ending': 'l', 'clitic': '', 'form': 'sg ad', 'partofspeech': 'N'}]),
Span('kaitseliitlasel', [{'normalized_text': 'kaitseliitlasel', 'lemma': 'kaitseliitlane', 'root': 'kaitse_liitlane', 'root_tokens': ['kaitse', 'liitlane'], 'ending': 'l', 'clitic': '', 'form': 'sg ad', 'partofspeech': 'S'}]),
Span(',', [{'normalized_text': ',', 'lemma': ',', 'root': ',', 'root_tokens': [','], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('politseinikul', [{'normalized_text': 'politseinikul', 'lemma': 'politseinik', 'root': 'politseinik', 'root_tokens': ['politseinik'], 'ending': 'l', 'clitic': '', 'form': 'sg ad', 'partofspeech': 'S'}]),
Span('ja', [{'normalized_text': 'ja', 'lemma': 'ja', 'root': 'ja', 'root_tokens': ['ja'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'J'}]),
Span('päästeametnikul', [{'normalized_text': 'päästeametnikul', 'lemma': 'päästeametnik', 'root': 'pääste_ametnik', 'root_tokens': ['pääste', 'ametnik'], 'ending': 'l', 'clitic': '', 'form': 'sg ad', 'partofspeech': 'S'}]),
Span('ei', [{'normalized_text': 'ei', 'lemma': 'ei', 'root': 'ei', 'root_tokens': ['ei'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}, {'normalized_text': 'ei', 'lemma': 'ei', 'root': 'ei', 'root_tokens': ['ei'], 'ending': '0', 'clitic': '', 'form': 'neg', 'partofspeech': 'V'}]),
Span('õnnestunud', [{'normalized_text': 'õnnestunud', 'lemma': 'õnnestuma', 'root': 'õnnestu', 'root_tokens': ['õnnestu'], 'ending': 'nud', 'clitic': '', 'form': 'nud', 'partofspeech': 'V'}, {'normalized_text': 'õnnestunud', 'lemma': 'õnnestunu', 'root': 'õnnestu=nu', 'root_tokens': ['õnnestunu'], 'ending': 'd', 'clitic': '', 'form': 'pl n', 'partofspeech': 'S'}, {'normalized_text': 'õnnestunud', 'lemma': 'õnnestunud', 'root': 'õnnestunud', 'root_tokens': ['õnnestunud'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'A'}, {'normalized_text': 'õnnestunud', 'lemma': 'õnnestunud', 'root': 'õnnestunud', 'root_tokens': ['õnnestunud'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'A'}, {'normalized_text': 'õnnestunud', 'lemma': 'õnnestunud', 'root': 'õnnestunud', 'root_tokens': ['õnnestunud'], 'ending': 'd', 'clitic': '', 'form': 'pl n', 'partofspeech': 'A'}]),
Span('läinud', [{'normalized_text': 'läinud', 'lemma': 'läinu', 'root': 'läinu', 'root_tokens': ['läinu'], 'ending': 'd', 'clitic': '', 'form': 'pl n', 'partofspeech': 'S'}, {'normalized_text': 'läinud', 'lemma': 'läinud', 'root': 'läinud', 'root_tokens': ['läinud'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'A'}, {'normalized_text': 'läinud', 'lemma': 'läinud', 'root': 'läinud', 'root_tokens': ['läinud'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'A'}, {'normalized_text': 'läinud', 'lemma': 'läinud', 'root': 'läinud', 'root_tokens': ['läinud'], 'ending': 'd', 'clitic': '', 'form': 'pl n', 'partofspeech': 'A'}, {'normalized_text': 'läinud', 'lemma': 'minema', 'root': 'mine', 'root_tokens': ['mine'], 'ending': 'nud', 'clitic': '', 'form': 'nud', 'partofspeech': 'V'}]),
Span('nädala', [{'normalized_text': 'nädala', 'lemma': 'nädal', 'root': 'nädal', 'root_tokens': ['nädal'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'S'}]),
Span('lõpuks', [{'normalized_text': 'lõpuks', 'lemma': 'lõpp', 'root': 'lõpp', 'root_tokens': ['lõpp'], 'ending': 'ks', 'clitic': '', 'form': 'sg tr', 'partofspeech': 'A'}, {'normalized_text': 'lõpuks', 'lemma': 'lõpp', 'root': 'lõpp', 'root_tokens': ['lõpp'], 'ending': 'ks', 'clitic': '', 'form': 'sg tr', 'partofspeech': 'S'}, {'normalized_text': 'lõpuks', 'lemma': 'lõpuks', 'root': 'lõpuks', 'root_tokens': ['lõpuks'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}]),
Span('leida', [{'no

In [11]:
text.stanza_syntax

Layer(name='stanza_syntax', attributes=('id', 'lemma', 'upostag', 'xpostag', 'feats', 'head', 'deprel', 'deps', 'misc'), spans=SL[Span('Poolesajal', [{'id': 1, 'lemma': 'poolsada', 'upostag': 'N', 'xpostag': 'N', 'feats': OrderedDict([('card', 'card'), ('sg', 'sg'), ('ad', 'ad'), ('l', 'l')]), 'head': 8, 'deprel': 'nsubj', 'deps': '_', 'misc': '_'}]),
Span('kaitseliitlasel', [{'id': 2, 'lemma': 'kaitseliitlane', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('ad', 'ad')]), 'head': 8, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span(',', [{'id': 3, 'lemma': ',', 'upostag': 'Z', 'xpostag': 'Z', 'feats': OrderedDict(), 'head': 4, 'deprel': 'punct', 'deps': '_', 'misc': '_'}]),
Span('politseinikul', [{'id': 4, 'lemma': 'politseinik', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('ad', 'ad')]), 'head': 2, 'deprel': 'conj', 'deps': '_', 'misc': '_'}]),
Span('ja', [{'id': 5, 'lemma': 'ja', 'upostag': 'J', 'xpostag': 'J', 'feats': OrderedDict([('sub', 'sub'), ('crd', 'crd')]), 'head': 6, 'deprel': 'cc', 'deps': '_', 'misc': '_'}]),
Span('päästeametnikul', [{'id': 6, 'lemma': 'päästeametnik', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('ad', 'ad')]), 'head': 2, 'deprel': 'conj', 'deps': '_', 'misc': '_'}]),
Span('ei', [{'id': 7, 'lemma': 'ei', 'upostag': 'D', 'xpostag': 'D', 'feats': OrderedDict(), 'head': 8, 'deprel': 'aux', 'deps': '_', 'misc': '_'}]),
Span('õnnestunud', [{'id': 8, 'lemma': 'õnnestuma', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('mod', 'mod'), ('partic', 'partic'), ('past', 'past'), ('ps', 'ps')]), 'head': 0, 'deprel': 'root', 'deps': '_', 'misc': '_'}]),
Span('läinud', [{'id': 9, 'lemma': 'minema', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('main', 'main'), ('partic', 'partic'), ('past', 'past'), ('ps', 'ps')]), 'head': 10, 'deprel': 'acl', 'deps': '_', 'misc': '_'}]),
Span('nädala', [{'id': 10, 'lemma': 'nädal', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('gen', 'gen')]), 'head': 12, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('lõpuks', [{'id': 11, 'lemma': 'lõpuks', 'upostag': 'D', 'xpostag': 'D', 'feats': OrderedDict(), 'head': 12, 'deprel': 'advmod', 'deps': '_', 'misc': '_'}]),
Span('leida', [{'id': 12, 'lemma': 'leidma', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('main', 'main'), ('inf', 'inf')]), 'head': 8, 'deprel': 'csubj', 'deps': '_', 'misc': '_'}]),
Span('Jõhvi', [{'id': 13, 'lemma': 'jõhv', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('part', 'part')]), 'head': 14, 'deprel': 'nmod', 'deps': '_', 'misc': '_'}]),
Span('lähedastest', [{'id': 14, 'lemma': 'lähedane', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('pl', 'pl'), ('el', 'el')]), 'head': 12, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('metsadest', [{'id': 15, 'lemma': 'mets', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('pl', 'pl'), ('el', 'el')]), 'head': 12, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('23-aastast', [{'id': 16, 'lemma': '23aastane', 'upostag': 'A', 'xpostag': 'A', 'feats': OrderedDict([('pos', 'pos'), ('sg', 'sg'), ('part', 'part')]), 'head': 17, 'deprel': 'amod', 'deps': '_', 'misc': '_'}]),
Span('neiut', [{'id': 17, 'lemma': 'neiu', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('part', 'part')]), 'head': 12, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span(',', [{'id': 18, 'lemma': ',', 'upostag': 'Z', 'xpostag': 'Z', 'feats': OrderedDict(), 'head': 20, 'deprel': 'punct', 'deps': '_', 'misc': '_'}]),
Span('kes', [{'id': 19, 'lemma': 'kes', 'upostag': 'P', 'xpostag': 'P', 'feats': OrderedDict([('pl', 'pl'), ('nom', 'nom')]), 'head': 20, 'deprel': 'nsubj', 'deps': '_', 'misc': '_'}]),
Span('varastas', [{'id': 20, 'lemma': 'varastama', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDic

In [120]:
ex = df1[df1["sentence"].str.contains("Sellegipoolest töötasid eelmisel aastal kokku 659")]
text, analysis, word_analysis, head_variants = get_analysis(ex)
print(text, "\n\nhead_word:", ex.iloc[0].form)
head_variants

Text(text='Sellegipoolest töötasid eelmisel aastal kokku 659 Tallinna noort vanuses 13-18 aastat 34 linnasiseses ja üheksas linnavälises rühmas .') 

head_word: vanuses


[('vanune', 'sg in', 'A'), ('vanus', 'sg in', 'S')]

In [75]:
analysis

Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('Sellegipoolest', [{'normalized_text': 'Sellegipoolest', 'lemma': 'sellegipoolest', 'root': 'sellegi_poolest', 'root_tokens': ['sellegi', 'poolest'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}]),
Span('töötasid', [{'normalized_text': 'töötasid', 'lemma': 'töötama', 'root': 'tööta', 'root_tokens': ['tööta'], 'ending': 'sid', 'clitic': '', 'form': 'sid', 'partofspeech': 'V'}]),
Span('eelmisel', [{'normalized_text': 'eelmisel', 'lemma': 'eelmine', 'root': 'eelmine', 'root_tokens': ['eelmine'], 'ending': 'l', 'clitic': '', 'form': 'sg ad', 'partofspeech': 'A'}]),
Span('aastal', [{'normalized_text': 'aastal', 'lemma': 'aasta', 'root': 'aasta', 'root_tokens': ['aasta'], 'ending': 'l', 'clitic': '', 'form': 'sg ad', 'partofspeech': 'S'}]),
Span('kokku', [{'normalized_text': 'kokku', 'lemma': 'kokku', 'root': 'kokku', 'root_tokens': ['kokku'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}, {'normalized_text': 'kokku', 'lemma': 'kogu', 'root': 'kogu', 'root_tokens': ['kogu'], 'ending': '0', 'clitic': '', 'form': 'adt', 'partofspeech': 'S'}]),
Span('659', [{'normalized_text': '659', 'lemma': '659', 'root': '659', 'root_tokens': ['659'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'N'}]),
Span('Tallinna', [{'normalized_text': 'Tallinna', 'lemma': 'Tallinn', 'root': 'Tallinn', 'root_tokens': ['Tallinn'], 'ending': '0', 'clitic': '', 'form': 'adt', 'partofspeech': 'H'}, {'normalized_text': 'Tallinna', 'lemma': 'Tallinn', 'root': 'Tallinn', 'root_tokens': ['Tallinn'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'H'}, {'normalized_text': 'Tallinna', 'lemma': 'tallinna', 'root': 'tallinna', 'root_tokens': ['tallinna'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'G'}, {'normalized_text': 'Tallinna', 'lemma': 'Tallinn', 'root': 'Tallinn', 'root_tokens': ['Tallinn'], 'ending': '0', 'clitic': '', 'form': 'sg p', 'partofspeech': 'H'}]),
Span('noort', [{'normalized_text': 'noort', 'lemma': 'noor', 'root': 'noor', 'root_tokens': ['noor'], 'ending': 't', 'clitic': '', 'form': 'sg p', 'partofspeech': 'A'}, {'normalized_text': 'noort', 'lemma': 'noor', 'root': 'noor', 'root_tokens': ['noor'], 'ending': 't', 'clitic': '', 'form': 'sg p', 'partofspeech': 'S'}]),
Span('vanuses', [{'normalized_text': 'vanuses', 'lemma': 'vanune', 'root': 'vanune', 'root_tokens': ['vanune'], 'ending': 's', 'clitic': '', 'form': 'sg in', 'partofspeech': 'A'}, {'normalized_text': 'vanuses', 'lemma': 'vanus', 'root': 'vanus', 'root_tokens': ['vanus'], 'ending': 's', 'clitic': '', 'form': 'sg in', 'partofspeech': 'S'}]),
Span('13', [{'normalized_text': '13', 'lemma': '13', 'root': '13', 'root_tokens': ['13'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'N'}]),
Span('-', [{'normalized_text': '-', 'lemma': '-', 'root': '-', 'root_tokens': ['-'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('18', [{'normalized_text': '18', 'lemma': '18', 'root': '18', 'root_tokens': ['18'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'N'}]),
Span('aastat', [{'normalized_text': 'aastat', 'lemma': 'aasta', 'root': 'aasta', 'root_tokens': ['aasta'], 'ending': 't', 'clitic': '', 'form': 'sg p', 'partofspeech': 'S'}]),
Span('34', [{'normalized_text': '34', 'lemma': '34', 'root': '34', 'root_tokens': ['34'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'N'}]),
Span('linnasiseses', [{'normalized_text': 'linnasiseses', 'lemma': 'linnasisene', 'root': 'linna_sisene', 'root_tokens': ['linna', 'sisene'], 'ending': 's', 'clitic': '', 'form': 'sg in', 'partofspeech': 'A'}]),
Span('ja', [{'normalized_text': 'ja', 'lemma': 'ja', 'root': 'ja', 'root_tokens': ['ja'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'J'}]),
Span('üheksas', [{'normalized_text': 'üheksas', 'lemma': 'üheksa', 'root': 'üheksa', 'root_tok

In [74]:
text.stanza_syntax

Layer(name='stanza_syntax', attributes=('id', 'lemma', 'upostag', 'xpostag', 'feats', 'head', 'deprel', 'deps', 'misc'), spans=SL[Span('Sellegipoolest', [{'id': 1, 'lemma': 'sellegipoolest', 'upostag': 'D', 'xpostag': 'D', 'feats': OrderedDict(), 'head': 2, 'deprel': 'advmod', 'deps': '_', 'misc': '_'}]),
Span('töötasid', [{'id': 2, 'lemma': 'töötama', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('main', 'main'), ('indic', 'indic'), ('impf', 'impf'), ('ps2', 'ps2'), ('sg', 'sg'), ('ps', 'ps'), ('af', 'af')]), 'head': 0, 'deprel': 'root', 'deps': '_', 'misc': '_'}]),
Span('eelmisel', [{'id': 3, 'lemma': 'eelmine', 'upostag': 'A', 'xpostag': 'A', 'feats': OrderedDict([('pos', 'pos'), ('sg', 'sg'), ('ad', 'ad')]), 'head': 4, 'deprel': 'amod', 'deps': '_', 'misc': '_'}]),
Span('aastal', [{'id': 4, 'lemma': 'aasta', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('ad', 'ad')]), 'head': 2, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('kokku', [{'id': 5, 'lemma': 'kogu', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('adit', 'adit')]), 'head': 2, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('659', [{'id': 6, 'lemma': '659', 'upostag': 'N', 'xpostag': 'N', 'feats': OrderedDict([('card', 'card'), ('<?>', '<?>'), ('digit', 'digit')]), 'head': 8, 'deprel': 'nummod', 'deps': '_', 'misc': '_'}]),
Span('Tallinna', [{'id': 7, 'lemma': 'Tallinn', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('prop', 'prop'), ('sg', 'sg'), ('gen', 'gen')]), 'head': 8, 'deprel': 'nmod', 'deps': '_', 'misc': '_'}]),
Span('noort', [{'id': 8, 'lemma': 'noor', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('part', 'part')]), 'head': 2, 'deprel': 'nsubj', 'deps': '_', 'misc': '_'}]),
Span('vanuses', [{'id': 9, 'lemma': 'vanus', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('in', 'in')]), 'head': 8, 'deprel': 'nmod', 'deps': '_', 'misc': '_'}]),
Span('13', [{'id': 10, 'lemma': '13', 'upostag': 'N', 'xpostag': 'N', 'feats': OrderedDict([('card', 'card'), ('<?>', '<?>'), ('digit', 'digit')]), 'head': 13, 'deprel': 'nummod', 'deps': '_', 'misc': '_'}]),
Span('-', [{'id': 11, 'lemma': '-', 'upostag': 'Z', 'xpostag': 'Z', 'feats': OrderedDict(), 'head': 12, 'deprel': 'punct', 'deps': '_', 'misc': '_'}]),
Span('18', [{'id': 12, 'lemma': '18', 'upostag': 'N', 'xpostag': 'N', 'feats': OrderedDict([('card', 'card'), ('<?>', '<?>'), ('digit', 'digit')]), 'head': 10, 'deprel': 'conj', 'deps': '_', 'misc': '_'}]),
Span('aastat', [{'id': 13, 'lemma': 'aasta', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('part', 'part')]), 'head': 2, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('34', [{'id': 14, 'lemma': '34', 'upostag': 'N', 'xpostag': 'N', 'feats': OrderedDict([('card', 'card'), ('<?>', '<?>'), ('digit', 'digit')]), 'head': 13, 'deprel': 'nummod', 'deps': '_', 'misc': '_'}]),
Span('linnasiseses', [{'id': 15, 'lemma': 'linnasisene', 'upostag': 'A', 'xpostag': 'A', 'feats': OrderedDict([('pos', 'pos'), ('sg', 'sg'), ('in', 'in')]), 'head': 13, 'deprel': 'nmod', 'deps': '_', 'misc': '_'}]),
Span('ja', [{'id': 16, 'lemma': 'ja', 'upostag': 'J', 'xpostag': 'J', 'feats': OrderedDict([('sub', 'sub'), ('crd', 'crd')]), 'head': 19, 'deprel': 'cc', 'deps': '_', 'misc': '_'}]),
Span('üheksas', [{'id': 17, 'lemma': 'üheksas', 'upostag': 'N', 'xpostag': 'N', 'feats': OrderedDict([('ord', 'ord'), ('sg', 'sg'), ('nom', 'nom'), ('roman', 'roman')]), 'head': 19, 'deprel': 'nsubj', 'deps': '_', 'misc': '_'}]),
Span('linnavälises', [{'id': 18, 'lemma': 'linnaväline', 'upostag': 'A', 'xpostag': 'A', 'feats': OrderedDict([('pos', 'pos'), ('sg', 'sg'), ('in', 'in')]), 'head': 19, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('rühmas', [{'id': 19, 'lemma': 'rühmama', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('main', 'main'), ('indic', 'indic'), ('impf', 'impf

In [8]:
ex = df1[df1["sentence"].str.contains("Peale selle , et paljude inimeste jaoks on rott vastik")]
text, analysis, word_analysis, head_variants = get_analysis(ex)
print(text, "\n\nhead_word:", ex.iloc[0].form)
head_variants

Text(text='Peale selle , et paljude inimeste jaoks on rott vastik , levitab elukas kõikvõimalikke haigusi .') 

head_word: elukas


[('elukas', 'sg n', 'S'), ('elukas', 'sg in', 'S')]

In [9]:
analysis

Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('Peale', [{'normalized_text': 'Peale', 'lemma': 'Pea', 'root': 'Pea', 'root_tokens': ['Pea'], 'ending': 'le', 'clitic': '', 'form': 'sg all', 'partofspeech': 'H'}, {'normalized_text': 'Peale', 'lemma': 'Peale', 'root': 'Peale', 'root_tokens': ['Peale'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'H'}, {'normalized_text': 'Peale', 'lemma': 'Peale', 'root': 'Peale', 'root_tokens': ['Peale'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'H'}, {'normalized_text': 'Peale', 'lemma': 'pea', 'root': 'pea', 'root_tokens': ['pea'], 'ending': 'le', 'clitic': '', 'form': 'sg all', 'partofspeech': 'S'}, {'normalized_text': 'Peale', 'lemma': 'peale', 'root': 'peale', 'root_tokens': ['peale'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}, {'normalized_text': 'Peale', 'lemma': 'peale', 'root': 'peale', 'root_tokens': ['peale'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'K'}]),
Span('selle', [{'normalized_text': 'selle', 'lemma': 'see', 'root': 'see', 'root_tokens': ['see'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'P'}, {'normalized_text': 'selle', 'lemma': 'sell', 'root': 'sell', 'root_tokens': ['sell'], 'ending': 'e', 'clitic': '', 'form': 'pl p', 'partofspeech': 'S'}]),
Span(',', [{'normalized_text': ',', 'lemma': ',', 'root': ',', 'root_tokens': [','], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('et', [{'normalized_text': 'et', 'lemma': 'et', 'root': 'et', 'root_tokens': ['et'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'J'}]),
Span('paljude', [{'normalized_text': 'paljude', 'lemma': 'palju', 'root': 'palju', 'root_tokens': ['palju'], 'ending': 'de', 'clitic': '', 'form': 'pl g', 'partofspeech': 'P'}]),
Span('inimeste', [{'normalized_text': 'inimeste', 'lemma': 'inimene', 'root': 'inimene', 'root_tokens': ['inimene'], 'ending': 'te', 'clitic': '', 'form': 'pl g', 'partofspeech': 'S'}]),
Span('jaoks', [{'normalized_text': 'jaoks', 'lemma': 'jaoks', 'root': 'jaoks', 'root_tokens': ['jaoks'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}, {'normalized_text': 'jaoks', 'lemma': 'jaoks', 'root': 'jaoks', 'root_tokens': ['jaoks'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'K'}, {'normalized_text': 'jaoks', 'lemma': 'jagu', 'root': 'jagu', 'root_tokens': ['jagu'], 'ending': 'ks', 'clitic': '', 'form': 'sg tr', 'partofspeech': 'S'}]),
Span('on', [{'normalized_text': 'on', 'lemma': 'olema', 'root': 'ole', 'root_tokens': ['ole'], 'ending': '0', 'clitic': '', 'form': 'b', 'partofspeech': 'V'}, {'normalized_text': 'on', 'lemma': 'olema', 'root': 'ole', 'root_tokens': ['ole'], 'ending': '0', 'clitic': '', 'form': 'vad', 'partofspeech': 'V'}]),
Span('rott', [{'normalized_text': 'rott', 'lemma': 'rott', 'root': 'rott', 'root_tokens': ['rott'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('vastik', [{'normalized_text': 'vastik', 'lemma': 'vastik', 'root': 'vastik', 'root_tokens': ['vastik'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'A'}]),
Span(',', [{'normalized_text': ',', 'lemma': ',', 'root': ',', 'root_tokens': [','], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('levitab', [{'normalized_text': 'levitab', 'lemma': 'levitama', 'root': 'levita', 'root_tokens': ['levita'], 'ending': 'b', 'clitic': '', 'form': 'b', 'partofspeech': 'V'}]),
Span('elukas', [{'normalized_text': 'elukas', 'lemma': 'elukas', 'root': 'elukas', 'root_tokens': ['elukas'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}, {'normalized_text': 'elukas', 'lemma': 'elukas', 'root': 'elukas', 'root_tokens': ['elukas'], 'ending': 's', 'clitic': '', 'form': 'sg in', 'partofspeech': 'S'}]),
Span('kõikvõimalikke', [{'normalized_text': 'kõikvõimalikke', 'lemma': 'kõikvõimalik', 'root': 'kõik_võimalik'

In [261]:
ex = df1[df1["sentence"].str.contains("Eriti just viimasest , sest kui Kreitzbergi")]
text, analysis, word_analysis, head_variants = get_analysis(ex)
print(text, "\n\nhead_word:", ex.iloc[0].form)
head_variants

Text(text='Eriti just viimasest , sest kui Kreitzbergi passis peaks olema perekonnaseisu kohta tempel " lahutatud " , siis tema elukaaslasel Sirje Priimäel ilutseb seal " abielus " .') 

head_word: abielus


[('abi_elu', 'sg in', 'S')]

In [154]:
ex = df1[df1["sentence"].str.contains("lätti")]
text, analysis, word_analysis, head_variants = get_analysis(ex)
print(text, "\n\nhead_word:", ex.iloc[0].form)
word_analysis

Text(text='kissy: jes ferru võtab mind tatu kontsertile lätti kaasa :D') 

head_word: lätti


text,normalized_text,lemma,root,root_tokens,ending,clitic,form,partofspeech
lätti,lätti,lätt,lätt,['lätt'],0,,adt,S
,lätti,lätt,lätt,['lätt'],0,,sg g,S
,lätti,lätt,lätt,['lätt'],0,,sg p,S
,lätti,lätti,lätti,['lätti'],0,,sg g,S
,lätti,lätti,lätti,['lätti'],0,,sg n,S
,lätti,lättima,lätti,['lätti'],0,,o,V


In [10]:
ex = df1[(df1["sentence"].str.contains("nurgas lamas hõbedases kumas loom")) & (df1["form"]=="hõbedases")]
text, analysis, word_analysis, head_variants = get_analysis(ex)
print(text, "\n\nhead_word:", ex.iloc[0].form)
analysis

Text(text='Ruumi kaugemais nurgas lamas hõbedases kumas loom , tema pea oli nukralt norgus ja tema suured pruunid silmad keerasid ükskõikselt tulijate poole .') 

head_word: hõbedases


Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('Ruumi', [{'normalized_text': 'Ruumi', 'lemma': 'Ruum', 'root': 'Ruum', 'root_tokens': ['Ruum'], 'ending': '0', 'clitic': '', 'form': 'adt', 'partofspeech': 'H'}, {'normalized_text': 'Ruumi', 'lemma': 'Ruum', 'root': 'Ruum', 'root_tokens': ['Ruum'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'H'}, {'normalized_text': 'Ruumi', 'lemma': 'Ruum', 'root': 'Ruum', 'root_tokens': ['Ruum'], 'ending': '0', 'clitic': '', 'form': 'sg p', 'partofspeech': 'H'}, {'normalized_text': 'Ruumi', 'lemma': 'Ruumi', 'root': 'Ruumi', 'root_tokens': ['Ruumi'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'H'}, {'normalized_text': 'Ruumi', 'lemma': 'Ruumi', 'root': 'Ruumi', 'root_tokens': ['Ruumi'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'H'}, {'normalized_text': 'Ruumi', 'lemma': 'ruum', 'root': 'ruum', 'root_tokens': ['ruum'], 'ending': '0', 'clitic': '', 'form': 'adt', 'partofspeech': 'S'}, {'normalized_text': 'Ruumi', 'lemma': 'ruum', 'root': 'ruum', 'root_tokens': ['ruum'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'S'}, {'normalized_text': 'Ruumi', 'lemma': 'ruumima', 'root': 'ruumi', 'root_tokens': ['ruumi'], 'ending': '0', 'clitic': '', 'form': 'o', 'partofspeech': 'V'}, {'normalized_text': 'Ruumi', 'lemma': 'ruum', 'root': 'ruum', 'root_tokens': ['ruum'], 'ending': '0', 'clitic': '', 'form': 'sg p', 'partofspeech': 'S'}]),
Span('kaugemais', [{'normalized_text': 'kaugemais', 'lemma': 'kaugem', 'root': 'kaugem', 'root_tokens': ['kaugem'], 'ending': 'is', 'clitic': '', 'form': 'pl in', 'partofspeech': 'C'}]),
Span('nurgas', [{'normalized_text': 'nurgas', 'lemma': 'nurk', 'root': 'nurk', 'root_tokens': ['nurk'], 'ending': 's', 'clitic': '', 'form': 'sg in', 'partofspeech': 'S'}]),
Span('lamas', [{'normalized_text': 'lamas', 'lemma': 'lamama', 'root': 'lama', 'root_tokens': ['lama'], 'ending': 's', 'clitic': '', 'form': 's', 'partofspeech': 'V'}]),
Span('hõbedases', [{'normalized_text': 'hõbedases', 'lemma': 'hõbedane', 'root': 'hõbedane', 'root_tokens': ['hõbedane'], 'ending': 's', 'clitic': '', 'form': 'sg in', 'partofspeech': 'A'}]),
Span('kumas', [{'normalized_text': 'kumas', 'lemma': 'kuma', 'root': 'kuma', 'root_tokens': ['kuma'], 'ending': 's', 'clitic': '', 'form': 'sg in', 'partofspeech': 'S'}, {'normalized_text': 'kumas', 'lemma': 'kumama', 'root': 'kuma', 'root_tokens': ['kuma'], 'ending': 's', 'clitic': '', 'form': 's', 'partofspeech': 'V'}]),
Span('loom', [{'normalized_text': 'loom', 'lemma': 'loom', 'root': 'loom', 'root_tokens': ['loom'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span(',', [{'normalized_text': ',', 'lemma': ',', 'root': ',', 'root_tokens': [','], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('tema', [{'normalized_text': 'tema', 'lemma': 'tema', 'root': 'tema', 'root_tokens': ['tema'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'P'}, {'normalized_text': 'tema', 'lemma': 'tema', 'root': 'tema', 'root_tokens': ['tema'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'P'}]),
Span('pea', [{'normalized_text': 'pea', 'lemma': 'pea', 'root': 'pea', 'root_tokens': ['pea'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}, {'normalized_text': 'pea', 'lemma': 'pea', 'root': 'pea', 'root_tokens': ['pea'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'S'}, {'normalized_text': 'pea', 'lemma': 'pidama', 'root': 'pida', 'root_tokens': ['pida'], 'ending': '0', 'clitic': '', 'form': 'o', 'partofspeech': 'V'}, {'normalized_text': 'pea', 'lemma': 'pea', 'root': 'pea', 'root_tokens': ['pea'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('oli', [{'normalized_text': 'oli', 'lemma': 'olema', 'root': 'ole', 'root_tokens': ['ole'], 'ending': 'i', 'clitic': '', 'form': 's', 'partofsp

In [11]:
analysis

Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('Ruumi', [{'normalized_text': 'Ruumi', 'lemma': 'Ruum', 'root': 'Ruum', 'root_tokens': ['Ruum'], 'ending': '0', 'clitic': '', 'form': 'adt', 'partofspeech': 'H'}, {'normalized_text': 'Ruumi', 'lemma': 'Ruum', 'root': 'Ruum', 'root_tokens': ['Ruum'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'H'}, {'normalized_text': 'Ruumi', 'lemma': 'Ruum', 'root': 'Ruum', 'root_tokens': ['Ruum'], 'ending': '0', 'clitic': '', 'form': 'sg p', 'partofspeech': 'H'}, {'normalized_text': 'Ruumi', 'lemma': 'Ruumi', 'root': 'Ruumi', 'root_tokens': ['Ruumi'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'H'}, {'normalized_text': 'Ruumi', 'lemma': 'Ruumi', 'root': 'Ruumi', 'root_tokens': ['Ruumi'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'H'}, {'normalized_text': 'Ruumi', 'lemma': 'ruum', 'root': 'ruum', 'root_tokens': ['ruum'], 'ending': '0', 'clitic': '', 'form': 'adt', 'partofspeech': 'S'}, {'normalized_text': 'Ruumi', 'lemma': 'ruum', 'root': 'ruum', 'root_tokens': ['ruum'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'S'}, {'normalized_text': 'Ruumi', 'lemma': 'ruumima', 'root': 'ruumi', 'root_tokens': ['ruumi'], 'ending': '0', 'clitic': '', 'form': 'o', 'partofspeech': 'V'}, {'normalized_text': 'Ruumi', 'lemma': 'ruum', 'root': 'ruum', 'root_tokens': ['ruum'], 'ending': '0', 'clitic': '', 'form': 'sg p', 'partofspeech': 'S'}]),
Span('kaugemais', [{'normalized_text': 'kaugemais', 'lemma': 'kaugem', 'root': 'kaugem', 'root_tokens': ['kaugem'], 'ending': 'is', 'clitic': '', 'form': 'pl in', 'partofspeech': 'C'}]),
Span('nurgas', [{'normalized_text': 'nurgas', 'lemma': 'nurk', 'root': 'nurk', 'root_tokens': ['nurk'], 'ending': 's', 'clitic': '', 'form': 'sg in', 'partofspeech': 'S'}]),
Span('lamas', [{'normalized_text': 'lamas', 'lemma': 'lamama', 'root': 'lama', 'root_tokens': ['lama'], 'ending': 's', 'clitic': '', 'form': 's', 'partofspeech': 'V'}]),
Span('hõbedases', [{'normalized_text': 'hõbedases', 'lemma': 'hõbedane', 'root': 'hõbedane', 'root_tokens': ['hõbedane'], 'ending': 's', 'clitic': '', 'form': 'sg in', 'partofspeech': 'A'}]),
Span('kumas', [{'normalized_text': 'kumas', 'lemma': 'kuma', 'root': 'kuma', 'root_tokens': ['kuma'], 'ending': 's', 'clitic': '', 'form': 'sg in', 'partofspeech': 'S'}, {'normalized_text': 'kumas', 'lemma': 'kumama', 'root': 'kuma', 'root_tokens': ['kuma'], 'ending': 's', 'clitic': '', 'form': 's', 'partofspeech': 'V'}]),
Span('loom', [{'normalized_text': 'loom', 'lemma': 'loom', 'root': 'loom', 'root_tokens': ['loom'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span(',', [{'normalized_text': ',', 'lemma': ',', 'root': ',', 'root_tokens': [','], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('tema', [{'normalized_text': 'tema', 'lemma': 'tema', 'root': 'tema', 'root_tokens': ['tema'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'P'}, {'normalized_text': 'tema', 'lemma': 'tema', 'root': 'tema', 'root_tokens': ['tema'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'P'}]),
Span('pea', [{'normalized_text': 'pea', 'lemma': 'pea', 'root': 'pea', 'root_tokens': ['pea'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}, {'normalized_text': 'pea', 'lemma': 'pea', 'root': 'pea', 'root_tokens': ['pea'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'S'}, {'normalized_text': 'pea', 'lemma': 'pidama', 'root': 'pida', 'root_tokens': ['pida'], 'ending': '0', 'clitic': '', 'form': 'o', 'partofspeech': 'V'}, {'normalized_text': 'pea', 'lemma': 'pea', 'root': 'pea', 'root_tokens': ['pea'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('oli', [{'normalized_text': 'oli', 'lemma': 'olema', 'root': 'ole', 'root_tokens': ['ole'], 'ending': 'i', 'clitic': '', 'form': 's', 'partofsp

In [202]:
ex = df1[df1["sentence"].str.contains("pankreases ja maksas paiknevad")]
text, analysis, word_analysis, head_variants = get_analysis(ex)
print(text, "\n\nhead_word:", ex.iloc[0].form)
analysis

Text(text='KT-uuringul viis kuud pärast ravi alustamist olid pankreases ja maksas paiknevad kolded endise suurusega .') 

head_word: KT-uuringul


Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('KT-uuringul', [{'normalized_text': 'KT-uuringul', 'lemma': 'KT-uuring', 'root': 'KT-uuring', 'root_tokens': ['KT', 'uuring'], 'ending': 'l', 'clitic': '', 'form': 'sg ad', 'partofspeech': 'S'}]),
Span('viis', [{'normalized_text': 'viis', 'lemma': 'viima', 'root': 'vii', 'root_tokens': ['vii'], 'ending': 's', 'clitic': '', 'form': 's', 'partofspeech': 'V'}, {'normalized_text': 'viis', 'lemma': 'viis', 'root': 'viis', 'root_tokens': ['viis'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'N'}, {'normalized_text': 'viis', 'lemma': 'viis', 'root': 'viis', 'root_tokens': ['viis'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('kuud', [{'normalized_text': 'kuud', 'lemma': 'kuu', 'root': 'kuu', 'root_tokens': ['kuu'], 'ending': 'd', 'clitic': '', 'form': 'pl n', 'partofspeech': 'S'}, {'normalized_text': 'kuud', 'lemma': 'kuu', 'root': 'kuu', 'root_tokens': ['kuu'], 'ending': 'd', 'clitic': '', 'form': 'sg p', 'partofspeech': 'S'}]),
Span('pärast', [{'normalized_text': 'pärast', 'lemma': 'pära', 'root': 'pära', 'root_tokens': ['pära'], 'ending': 'st', 'clitic': '', 'form': 'sg el', 'partofspeech': 'S'}, {'normalized_text': 'pärast', 'lemma': 'pärane', 'root': 'pärane', 'root_tokens': ['pärane'], 'ending': 't', 'clitic': '', 'form': 'sg p', 'partofspeech': 'A'}, {'normalized_text': 'pärast', 'lemma': 'pärast', 'root': 'pärast', 'root_tokens': ['pärast'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}, {'normalized_text': 'pärast', 'lemma': 'pärast', 'root': 'pärast', 'root_tokens': ['pärast'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'K'}]),
Span('ravi', [{'normalized_text': 'ravi', 'lemma': 'ravi', 'root': 'ravi', 'root_tokens': ['ravi'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'S'}, {'normalized_text': 'ravi', 'lemma': 'ravima', 'root': 'ravi', 'root_tokens': ['ravi'], 'ending': '0', 'clitic': '', 'form': 'o', 'partofspeech': 'V'}, {'normalized_text': 'ravi', 'lemma': 'ravi', 'root': 'ravi', 'root_tokens': ['ravi'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}, {'normalized_text': 'ravi', 'lemma': 'ravi', 'root': 'ravi', 'root_tokens': ['ravi'], 'ending': '0', 'clitic': '', 'form': 'sg p', 'partofspeech': 'S'}]),
Span('alustamist', [{'normalized_text': 'alustamist', 'lemma': 'alustamine', 'root': 'alustamine', 'root_tokens': ['alustamine'], 'ending': 't', 'clitic': '', 'form': 'sg p', 'partofspeech': 'S'}]),
Span('olid', [{'normalized_text': 'olid', 'lemma': 'olema', 'root': 'ole', 'root_tokens': ['ole'], 'ending': 'id', 'clitic': '', 'form': 'sid', 'partofspeech': 'V'}]),
Span('pankreases', [{'normalized_text': 'pankreases', 'lemma': 'pankreas', 'root': 'pankreas', 'root_tokens': ['pankreas'], 'ending': 's', 'clitic': '', 'form': 'sg in', 'partofspeech': 'S'}]),
Span('ja', [{'normalized_text': 'ja', 'lemma': 'ja', 'root': 'ja', 'root_tokens': ['ja'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'J'}]),
Span('maksas', [{'normalized_text': 'maksas', 'lemma': 'maks', 'root': 'maks', 'root_tokens': ['maks'], 'ending': 's', 'clitic': '', 'form': 'sg in', 'partofspeech': 'S'}]),
Span('paiknevad', [{'normalized_text': 'paiknevad', 'lemma': 'paiknema', 'root': 'paikne', 'root_tokens': ['paikne'], 'ending': 'vad', 'clitic': '', 'form': 'vad', 'partofspeech': 'V'}, {'normalized_text': 'paiknevad', 'lemma': 'paiknev', 'root': 'paiknev', 'root_tokens': ['paiknev'], 'ending': 'd', 'clitic': '', 'form': 'pl n', 'partofspeech': 'A'}]),
Span('kolded', [{'normalized_text': 'kolded', 'lemma': 'kolle', 'root': 'kolle', 'root_tokens': ['kolle'], 'ending': 'd', 'clitic': '', 'form': 'pl n', 'partofspeech': 'S'}]),
Span('endise', [{'normalized_text': 'endise', 'lemma': 'endine', 'root': 'endine', 'root_tokens': ['endine'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeec

In [204]:
text.morph_extended

Layer(name='morph_extended', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech', 'punctuation_type', 'pronoun_type', 'letter_case', 'fin', 'verb_extension_suffix', 'subcat'), spans=SL[Span('KT-uuringul', [{'normalized_text': 'KT-uuringul', 'lemma': 'KT-uuring', 'root': 'KT-uuring', 'root_tokens': ['KT', 'uuring'], 'ending': 'l', 'clitic': '', 'form': 'com sg ad', 'partofspeech': 'S', 'punctuation_type': None, 'pronoun_type': None, 'letter_case': 'cap', 'fin': None, 'verb_extension_suffix': [], 'subcat': None}]),
Span('viis', [{'normalized_text': 'viis', 'lemma': 'viima', 'root': 'vii', 'root_tokens': ['vii'], 'ending': 's', 'clitic': '', 'form': 'mod indic impf ps3 sg ps af', 'partofspeech': 'V', 'punctuation_type': None, 'pronoun_type': None, 'letter_case': None, 'fin': True, 'verb_extension_suffix': [], 'subcat': ['NGP-P']}, {'normalized_text': 'viis', 'lemma': 'viima', 'root': 'vii', 'root_tokens': ['vii'], 'ending': 's', 'clitic': '', 'form': 'aux indic impf ps3 sg ps af', 'partofspeech': 'V', 'punctuation_type': None, 'pronoun_type': None, 'letter_case': None, 'fin': True, 'verb_extension_suffix': [], 'subcat': ['NGP-P']}, {'normalized_text': 'viis', 'lemma': 'viima', 'root': 'vii', 'root_tokens': ['vii'], 'ending': 's', 'clitic': '', 'form': 'main indic impf ps3 sg ps af', 'partofspeech': 'V', 'punctuation_type': None, 'pronoun_type': None, 'letter_case': None, 'fin': True, 'verb_extension_suffix': [], 'subcat': ['NGP-P']}, {'normalized_text': 'viis', 'lemma': 'viis', 'root': 'viis', 'root_tokens': ['viis'], 'ending': '0', 'clitic': '', 'form': 'card sg nom l', 'partofspeech': 'N', 'punctuation_type': None, 'pronoun_type': None, 'letter_case': None, 'fin': None, 'verb_extension_suffix': [], 'subcat': None}, {'normalized_text': 'viis', 'lemma': 'viis', 'root': 'viis', 'root_tokens': ['viis'], 'ending': '0', 'clitic': '', 'form': 'com sg nom', 'partofspeech': 'S', 'punctuation_type': None, 'pronoun_type': None, 'letter_case': None, 'fin': None, 'verb_extension_suffix': [], 'subcat': None}]),
Span('kuud', [{'normalized_text': 'kuud', 'lemma': 'kuu', 'root': 'kuu', 'root_tokens': ['kuu'], 'ending': 'd', 'clitic': '', 'form': 'com pl nom', 'partofspeech': 'S', 'punctuation_type': None, 'pronoun_type': None, 'letter_case': None, 'fin': None, 'verb_extension_suffix': [], 'subcat': None}, {'normalized_text': 'kuud', 'lemma': 'kuu', 'root': 'kuu', 'root_tokens': ['kuu'], 'ending': 'd', 'clitic': '', 'form': 'com sg part', 'partofspeech': 'S', 'punctuation_type': None, 'pronoun_type': None, 'letter_case': None, 'fin': None, 'verb_extension_suffix': [], 'subcat': None}]),
Span('pärast', [{'normalized_text': 'pärast', 'lemma': 'pära', 'root': 'pära', 'root_tokens': ['pära'], 'ending': 'st', 'clitic': '', 'form': 'com sg el', 'partofspeech': 'S', 'punctuation_type': None, 'pronoun_type': None, 'letter_case': None, 'fin': None, 'verb_extension_suffix': [], 'subcat': None}, {'normalized_text': 'pärast', 'lemma': 'pärane', 'root': 'pärane', 'root_tokens': ['pärane'], 'ending': 't', 'clitic': '', 'form': 'pos sg part', 'partofspeech': 'A', 'punctuation_type': None, 'pronoun_type': None, 'letter_case': None, 'fin': None, 'verb_extension_suffix': [], 'subcat': None}, {'normalized_text': 'pärast', 'lemma': 'pärast', 'root': 'pärast', 'root_tokens': ['pärast'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D', 'punctuation_type': None, 'pronoun_type': None, 'letter_case': None, 'fin': None, 'verb_extension_suffix': [], 'subcat': None}, {'normalized_text': 'pärast', 'lemma': 'pärast', 'root': 'pärast', 'root_tokens': ['pärast'], 'ending': '0', 'clitic': '', 'form': 'post', 'partofspeech': 'K', 'punctuation_type': None, 'pronoun_type': None, 'letter_case': None, 'fin': None, 'verb_extension_suffix': [], 'subcat': ['gen']}]),
Span('ravi', [{'normalized_text': 'ravi', 'lemma': 'ravi', 'root': 'ravi', 'root_tokens': ['ravi'], 'ending': '0', 'clitic': '', 'form': 'com sg gen', 'par

In [205]:
text.stanza_syntax

Layer(name='stanza_syntax', attributes=('id', 'lemma', 'upostag', 'xpostag', 'feats', 'head', 'deprel', 'deps', 'misc'), spans=SL[Span('KT-uuringul', [{'id': 1, 'lemma': 'KT-uuring', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('ad', 'ad')]), 'head': 14, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('viis', [{'id': 2, 'lemma': 'viis', 'upostag': 'N', 'xpostag': 'N', 'feats': OrderedDict([('card', 'card'), ('sg', 'sg'), ('nom', 'nom'), ('l', 'l')]), 'head': 3, 'deprel': 'nummod', 'deps': '_', 'misc': '_'}]),
Span('kuud', [{'id': 3, 'lemma': 'kuu', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('pl', 'pl'), ('nom', 'nom')]), 'head': 5, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('pärast', [{'id': 4, 'lemma': 'pärane', 'upostag': 'A', 'xpostag': 'A', 'feats': OrderedDict([('pos', 'pos'), ('sg', 'sg'), ('part', 'part')]), 'head': 5, 'deprel': 'amod', 'deps': '_', 'misc': '_'}]),
Span('ravi', [{'id': 5, 'lemma': 'ravima', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('mod', 'mod'), ('imper', 'imper'), ('pres', 'pres'), ('ps2', 'ps2'), ('sg', 'sg'), ('ps', 'ps'), ('neg', 'neg')]), 'head': 6, 'deprel': 'acl', 'deps': '_', 'misc': '_'}]),
Span('alustamist', [{'id': 6, 'lemma': 'alustamine', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('part', 'part')]), 'head': 14, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('olid', [{'id': 7, 'lemma': 'olema', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('main', 'main'), ('indic', 'indic'), ('impf', 'impf'), ('ps2', 'ps2'), ('sg', 'sg'), ('ps', 'ps'), ('af', 'af')]), 'head': 14, 'deprel': 'cop', 'deps': '_', 'misc': '_'}]),
Span('pankreases', [{'id': 8, 'lemma': 'pankreas', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('in', 'in')]), 'head': 11, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('ja', [{'id': 9, 'lemma': 'ja', 'upostag': 'J', 'xpostag': 'J', 'feats': OrderedDict([('sub', 'sub'), ('crd', 'crd')]), 'head': 10, 'deprel': 'cc', 'deps': '_', 'misc': '_'}]),
Span('maksas', [{'id': 10, 'lemma': 'maks', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('in', 'in')]), 'head': 8, 'deprel': 'conj', 'deps': '_', 'misc': '_'}]),
Span('paiknevad', [{'id': 11, 'lemma': 'paiknev', 'upostag': 'A', 'xpostag': 'A', 'feats': OrderedDict([('pos', 'pos'), ('pl', 'pl'), ('nom', 'nom')]), 'head': 12, 'deprel': 'acl', 'deps': '_', 'misc': '_'}]),
Span('kolded', [{'id': 12, 'lemma': 'kolle', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('pl', 'pl'), ('nom', 'nom')]), 'head': 14, 'deprel': 'nsubj:cop', 'deps': '_', 'misc': '_'}]),
Span('endise', [{'id': 13, 'lemma': 'endine', 'upostag': 'A', 'xpostag': 'A', 'feats': OrderedDict([('pos', 'pos'), ('sg', 'sg'), ('gen', 'gen')]), 'head': 14, 'deprel': 'amod', 'deps': '_', 'misc': '_'}]),
Span('suurusega', [{'id': 14, 'lemma': 'suurune', 'upostag': 'A', 'xpostag': 'A', 'feats': OrderedDict([('pos', 'pos'), ('sg', 'sg'), ('kom', 'kom')]), 'head': 0, 'deprel': 'root', 'deps': '_', 'misc': '_'}]),
Span('.', [{'id': 15, 'lemma': '.', 'upostag': 'Z', 'xpostag': 'Z', 'feats': OrderedDict(), 'head': 14, 'deprel': 'punct', 'deps': '_', 'misc': '_'}])])

In [213]:
# peasõna on verb
ex = df1[df1["sentence"].str.contains("Nii käitudes kaevavad riigi tagant")]
text, analysis, word_analysis, head_variants = get_analysis(ex)
print(text, "\n\nhead_word:", ex.iloc[0].form)
word_analysis

Text(text='Nii käitudes kaevavad riigi tagant varastavad kalamehed Puuritsa arvates iseendale auku .') 

head_word: käitudes


text,normalized_text,lemma,root,root_tokens,ending,clitic,form,partofspeech
käitudes,käitudes,käit,käit,['käit'],des,,pl in,S
,käitudes,käituma,käitu,['käitu'],des,,des,V


In [124]:
# nimisõna (kandis) on verbiks
narva_ex = df1[df1["sentence"].str.contains("Narva kandis")]
text, analysis, word_analysis, head_variants = get_analysis(narva_ex)
print(text, "\n\nhead_word:", narva_ex.iloc[0].form)
analysis

Text(text='Ta oli tulnud kuu-kahe eest Narva kandis üle piiri , kaasas Soome margad ja dollarid .') 

head_word: Narva


Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('Ta', [{'normalized_text': 'Ta', 'lemma': 'tema', 'root': 'tema', 'root_tokens': ['tema'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'P'}, {'normalized_text': 'Ta', 'lemma': 'tema', 'root': 'tema', 'root_tokens': ['tema'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'P'}]),
Span('oli', [{'normalized_text': 'oli', 'lemma': 'olema', 'root': 'ole', 'root_tokens': ['ole'], 'ending': 'i', 'clitic': '', 'form': 's', 'partofspeech': 'V'}]),
Span('tulnud', [{'normalized_text': 'tulnud', 'lemma': 'tulnud', 'root': 'tul=nud', 'root_tokens': ['tulnud'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'A'}, {'normalized_text': 'tulnud', 'lemma': 'tulnud', 'root': 'tul=nud', 'root_tokens': ['tulnud'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'A'}, {'normalized_text': 'tulnud', 'lemma': 'tulnud', 'root': 'tul=nud', 'root_tokens': ['tulnud'], 'ending': 'd', 'clitic': '', 'form': 'pl n', 'partofspeech': 'A'}, {'normalized_text': 'tulnud', 'lemma': 'tulema', 'root': 'tule', 'root_tokens': ['tule'], 'ending': 'nud', 'clitic': '', 'form': 'nud', 'partofspeech': 'V'}, {'normalized_text': 'tulnud', 'lemma': 'tulnu', 'root': 'tulnu', 'root_tokens': ['tulnu'], 'ending': 'd', 'clitic': '', 'form': 'pl n', 'partofspeech': 'S'}]),
Span('kuu-kahe', [{'normalized_text': 'kuu-kahe', 'lemma': 'kuu-kaks', 'root': 'kuu-kaks', 'root_tokens': ['kuu', 'kaks'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'N'}]),
Span('eest', [{'normalized_text': 'eest', 'lemma': 'eest', 'root': 'eest', 'root_tokens': ['eest'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}, {'normalized_text': 'eest', 'lemma': 'eest', 'root': 'eest', 'root_tokens': ['eest'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'K'}, {'normalized_text': 'eest', 'lemma': 'esi', 'root': 'esi', 'root_tokens': ['esi'], 'ending': 'st', 'clitic': '', 'form': 'sg el', 'partofspeech': 'S'}]),
Span('Narva', [{'normalized_text': 'Narva', 'lemma': 'Narva', 'root': 'Narva', 'root_tokens': ['Narva'], 'ending': '0', 'clitic': '', 'form': 'adt', 'partofspeech': 'H'}, {'normalized_text': 'Narva', 'lemma': 'Narva', 'root': 'Narva', 'root_tokens': ['Narva'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'H'}, {'normalized_text': 'Narva', 'lemma': 'Narva', 'root': 'Narva', 'root_tokens': ['Narva'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'H'}, {'normalized_text': 'Narva', 'lemma': 'narva', 'root': 'narva', 'root_tokens': ['narva'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'G'}]),
Span('kandis', [{'normalized_text': 'kandis', 'lemma': 'kant', 'root': 'kant', 'root_tokens': ['kant'], 'ending': 's', 'clitic': '', 'form': 'sg in', 'partofspeech': 'S'}, {'normalized_text': 'kandis', 'lemma': 'kandma', 'root': 'kand', 'root_tokens': ['kand'], 'ending': 'is', 'clitic': '', 'form': 's', 'partofspeech': 'V'}, {'normalized_text': 'kandis', 'lemma': 'kandis', 'root': 'kandis', 'root_tokens': ['kandis'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'K'}]),
Span('üle', [{'normalized_text': 'üle', 'lemma': 'üle', 'root': 'üle', 'root_tokens': ['üle'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}, {'normalized_text': 'üle', 'lemma': 'üle', 'root': 'üle', 'root_tokens': ['üle'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'K'}]),
Span('piiri', [{'normalized_text': 'piiri', 'lemma': 'piir', 'root': 'piir', 'root_tokens': ['piir'], 'ending': '0', 'clitic': '', 'form': 'adt', 'partofspeech': 'S'}, {'normalized_text': 'piiri', 'lemma': 'piir', 'root': 'piir', 'root_tokens': ['piir'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'S'}, {'normalized_text': 'piiri', 'lemma': 'piir', 'root': 'piir', 'root_tokens': ['piir'], 'ending': '0', 'clitic': '', 'form': 'sg p', 'partofspeech': 

In [72]:
text.stanza_syntax

Layer(name='stanza_syntax', attributes=('id', 'lemma', 'upostag', 'xpostag', 'feats', 'head', 'deprel', 'deps', 'misc'), spans=SL[Span('Ta', [{'id': 1, 'lemma': 'tema', 'upostag': 'P', 'xpostag': 'P', 'feats': OrderedDict([('sg', 'sg'), ('nom', 'nom')]), 'head': 3, 'deprel': 'nsubj', 'deps': '_', 'misc': '_'}]),
Span('oli', [{'id': 2, 'lemma': 'olema', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('mod', 'mod'), ('indic', 'indic'), ('impf', 'impf'), ('ps3', 'ps3'), ('sg', 'sg'), ('ps', 'ps'), ('af', 'af')]), 'head': 3, 'deprel': 'aux', 'deps': '_', 'misc': '_'}]),
Span('tulnud', [{'id': 3, 'lemma': 'tulema', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('mod', 'mod'), ('indic', 'indic'), ('impf', 'impf'), ('ps', 'ps'), ('neg', 'neg')]), 'head': 0, 'deprel': 'root', 'deps': '_', 'misc': '_'}]),
Span('kuu-kahe', [{'id': 4, 'lemma': 'kuu-kaks', 'upostag': 'N', 'xpostag': 'N', 'feats': OrderedDict([('card', 'card'), ('sg', 'sg'), ('gen', 'gen'), ('l', 'l')]), 'head': 3, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('eest', [{'id': 5, 'lemma': 'eest', 'upostag': 'K', 'xpostag': 'K', 'feats': OrderedDict([('post', 'post')]), 'head': 4, 'deprel': 'case', 'deps': '_', 'misc': '_'}]),
Span('Narva', [{'id': 6, 'lemma': 'Narva', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('prop', 'prop'), ('sg', 'sg'), ('gen', 'gen')]), 'head': 7, 'deprel': 'nmod', 'deps': '_', 'misc': '_'}]),
Span('kandis', [{'id': 7, 'lemma': 'kandma', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('mod', 'mod'), ('indic', 'indic'), ('impf', 'impf'), ('ps3', 'ps3'), ('sg', 'sg'), ('ps', 'ps'), ('af', 'af')]), 'head': 3, 'deprel': 'conj', 'deps': '_', 'misc': '_'}]),
Span('üle', [{'id': 8, 'lemma': 'üle', 'upostag': 'D', 'xpostag': 'D', 'feats': OrderedDict(), 'head': 9, 'deprel': 'case', 'deps': '_', 'misc': '_'}]),
Span('piiri', [{'id': 9, 'lemma': 'piir', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('part', 'part')]), 'head': 7, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span(',', [{'id': 10, 'lemma': ',', 'upostag': 'Z', 'xpostag': 'Z', 'feats': OrderedDict(), 'head': 11, 'deprel': 'punct', 'deps': '_', 'misc': '_'}]),
Span('kaasas', [{'id': 11, 'lemma': 'kaasas', 'upostag': 'D', 'xpostag': 'D', 'feats': OrderedDict(), 'head': 3, 'deprel': 'conj', 'deps': '_', 'misc': '_'}]),
Span('Soome', [{'id': 12, 'lemma': 'soome', 'upostag': 'G', 'xpostag': 'G', 'feats': OrderedDict(), 'head': 13, 'deprel': 'amod', 'deps': '_', 'misc': '_'}]),
Span('margad', [{'id': 13, 'lemma': 'mark', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('pl', 'pl'), ('nom', 'nom')]), 'head': 11, 'deprel': 'nsubj:cop', 'deps': '_', 'misc': '_'}]),
Span('ja', [{'id': 14, 'lemma': 'ja', 'upostag': 'J', 'xpostag': 'J', 'feats': OrderedDict([('sub', 'sub'), ('crd', 'crd')]), 'head': 15, 'deprel': 'cc', 'deps': '_', 'misc': '_'}]),
Span('dollarid', [{'id': 15, 'lemma': 'dollar', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('pl', 'pl'), ('nom', 'nom')]), 'head': 13, 'deprel': 'conj', 'deps': '_', 'misc': '_'}]),
Span('.', [{'id': 16, 'lemma': '.', 'upostag': 'Z', 'xpostag': 'Z', 'feats': OrderedDict(), 'head': 3, 'deprel': 'punct', 'deps': '_', 'misc': '_'}])])

## -vad lõpulised verbid

In [206]:
def get_analysis3(ex):
    txt = Text(ex["sentence"])
    txt.tag_layer("words")
    txt.tag_layer("sentences")
    morf_tagger.tag(txt)
    txt.tag_layer('morph_extended')
    stanza_tagger.tag( txt )
    
    head_loc = int(ex["head_loc"])-1
    head_analysis = txt.morph_analysis[head_loc]
    
    roots = list(head_analysis.root)
    lemmas = list(head_analysis.lemma)
    forms = list(head_analysis.form)
    poss = list(head_analysis.partofspeech)

    variants = []
    for r, f, pos in zip(lemmas, forms, poss):
        variants.append((r, f, pos))

    return txt, txt.morph_analysis, head_analysis, variants

In [207]:
df_no1 = df1[df1["classification2"]=="no"]

In [247]:
vad_data = []

for i in tqdm(range(len(df_no))):
    #ex = df1[df1["sentence"].str.contains("Raudteelaste kraesse toimunud")]
    ex = df_no1.iloc[i]
    #print(ex)
    #print(ex["sentence"])
    text, analysis, word_analysis, head_variants = get_analysis3(ex)
    
    analysis2 = text.morph_extended
    analysis3 = text.stanza_syntax
    
    verb = ex["verb"]
    
    verb_idx = 0
    for i, sublist in enumerate(list(analysis.lemma)): #.index("paiknema")
        if verb in sublist:
            verb_idx = i
            break
       
    #if verb == "paiknema":
    #    print(ex["sentence"])
    #    print(analysis[i].ending)
    
    
    form_elem = list(analysis2[i].form)
    pos = list(analysis[i].partofspeech)
    verb_idx = pos.index("V")
    
    #has_form = False
    #if "pres" in form_elem[verb_idx] and "aux" in form_elem[verb_idx] and "act" in form_elem[verb_idx]:
    #    has_form = True
        
    # kui need on verbi formis siis ei saa olle v-kesksõna sest on pöördeline või abiverb -gpt
    #v_kesk = True
    #formslst = form_elem[verb_idx].split(" ")
    #if any(x in form_elem[verb_idx] for x in ['aux', 'ind', 'imp', 'cond']):
    #    v_kesk = False
    
    # kas allub nimisõnale
    ylemuse_idx = analysis3[i].head
    ylemuse_pos = analysis[ylemuse_idx].partofspeech[0]
    is_noun = ylemuse_pos in ["S", "H"]
        
        
    # kas on -vad või -v lõpuline 
    if (list(analysis[i].ending)[verb_idx] == "vad" or list(analysis[i].ending)[verb_idx]=="v") and is_noun:
        #print("pres" in form_elem[verb_idx])
        vad_data.append(f"{list(analysis[i].ending)[verb_idx]}, {ylemuse_pos}") #",".join(analysis[i].ending)
    else:
        vad_data.append("")
    

100%|███████████████████████████████████████| 1306/1306 [00:41<00:00, 31.46it/s]


In [248]:
df_no1["vad_ending"] = vad_data

/tmp/ipykernel_32390/466822601.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_no1["vad_ending"] = vad_data


In [249]:
df_no1.to_csv(RESULT_DIR+ "n80_examples_large_v01_gpt_v02_10K_b10_v01_no_verb_vad_ending.csv", encoding="utf-8", index=False, sep=",", quoting=csv.QUOTE_MINIMAL)

In [254]:
vad = df_no1[["sentence_id","head_id", "head_loc", "verb", "verb_compound", "vad_ending",
              "morph_case", "lemma","form", "sentence"]]

vad = vad[vad["vad_ending"]!=""] # 61
vad

,sentence_id,head_id,head_loc,verb,verb_compound,vad_ending,morph_case,lemma,form,sentence
190,404048,628673,6,juhatama,NaN,"vad, H",in,seelik,seelikutes,Sajandi viimasel kümnendil juhatavad triibulistes seelikutes lõbusad mehikesed eesti spordipubliku taas Kadriorgu jalgpalli vaatama .
359,10269466,16479712,14,pidama,NaN,"vad, S",adit,arsenal,Arsenali,B-grupis peavad asjatundjad soosikuteks Itaalia meistrit Rooma Laziot ja UEFA karikasarja finalisti Londoni Arsenali .
551,568974,900261,4,pidama,NaN,"vad, H",adit,USA-visiit,USA-visiiti,Poliitikud peavad kaitseministri USA-visiiti Eestit häbistavaks
568,15208598,23747758,3,paistma,NaN,"vad, S",abl,defau,default,"actionbarid paistavad default WoWi omad olema , lisatud on Bottom Left ja Bottom Right Barid ( Interface -> Actionbars )"
786,10828071,17342607,9,tervitama,NaN,"vad, S",in,laad,laadis,Oma uusi saastavendi tervitavad vanemad olijad väga iseloomulikus laadis :
...,...,...,...,...,...,...,...,...,...,...
9040,14312924,22721401,8,pidama,NaN,"vad, H",adit,kamp,kampa,Leppimatut vihavaenu trotsides peavad mehed pääsemise nimel kampa hakkama .
9105,11824608,18924355,1,kuuluma,NaN,"vad, H",in,America,America ’ s,America ’ s Cupi juurde kuuluvad tavapäraselt skandaalid .
9221,10211788,16386266,13,kõndima,NaN,"vad, S",in,diskopepupüks,diskopepupükstes,"Seda huvitavam on jälgida , kuidas läbi müravate algklassiõpilaste kõnnivad modelsel kõnnakul diskopepupükstes abituuriumieelsed tüdrukud ."
9703,1378240,2191848,6,saama,NaN,"vad, H",ill,HIV,HIV-sse,Narvas saavad positiivse testitulemuse 5 HIV-sse nakatunut .


In [255]:
vad.to_csv(RESULT_DIR+ "n80_examples_large_v01_gpt_v02_10K_b10_v01_no_verb_vad_ending_readable.csv", encoding="utf-8", index=False, sep=",", quoting=csv.QUOTE_MINIMAL)

## leida vormihomonüümid

ehk variants listis peaks olema rohkem kui 1

In [265]:
df_no = df1[df1["classification2"]=="no"]

In [266]:
def get_analysis2(ex):
    txt = Text(ex["sentence"])
    txt.tag_layer("words")
    txt.tag_layer("sentences")
    morf_tagger.tag(txt)
    txt.tag_layer('morph_extended')
    stanza_tagger.tag( txt )
    
    head_loc = int(ex["head_loc"])-1
    head_analysis = txt.morph_analysis[head_loc]
    
    roots = list(head_analysis.root)
    lemmas = list(head_analysis.lemma)
    forms = list(head_analysis.form)
    poss = list(head_analysis.partofspeech)

    variants = []
    for r, f, pos in zip(lemmas, forms, poss):
        variants.append((r, f, pos))

    return txt, txt.morph_analysis, head_analysis, variants

In [267]:
new_col_data = []

for i in tqdm(range(len(df_no))):
    #ex = df1[df1["sentence"].str.contains("Raudteelaste kraesse toimunud")]
    ex = df_no.iloc[i]
    #print(ex)
    #print(ex["sentence"])
    text, analysis, word_analysis, head_variants = get_analysis2(ex)
    if len(head_variants) > 1:
        #print(i, text, "\nhead_word:", ex["form"])
        #print(head_variants, "\n\n")
        new_col_data.append(head_variants)
        #break
    else:
        new_col_data.append("")

100%|███████████████████████████████████████| 1306/1306 [00:40<00:00, 32.33it/s]


In [268]:
df_no["head_variants"] = new_col_data

/tmp/ipykernel_32390/1856633017.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_no["head_variants"] = new_col_data


In [269]:
df_no.to_csv(RESULT_DIR+ "n80_examples_large_v01_gpt_v02_10K_b10_v01_no_head_variants.csv", encoding="utf-8", index=False, sep=",", quoting=csv.QUOTE_MINIMAL)

In [ ]:
df_no = pd.read_csv(RESULT_DIR+ "n80_examples_large_v01_gpt_v02_10K_b10_v01_no_head_variants.csv", encoding="utf-8", sep=",")

In [270]:
df_no[df_no["head_variants"]!= ""] # 515

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,classification,explanation,classification2,explanation2,timex_tag,ekilex_tag,ner_tag,head_variants
8,11102573,17784543,6,ravima,NaN,in,seis,seisus,"Kaks kuud ravis perenaine armetus seisus koera , kaotamata siiski lootust .",no,The phrase 'seisus' refers to 'condition' or 'state' and does not signify a location.,no,"The phrase 'seisus' refers to a condition or state rather than a location, so it is not adverbial of place.",NaN,NaN,NaN,"[(seis, sg in, S), (seisus, sg n, S)]"
27,15500381,24205992,12,panema,NaN,adit,hooldeprojekt,hooldeprojekti,""" Vajaduse korral panevad Veerpalu ja loodetavasti õed Šmigunid isiklikku raha hooldeprojekti , sest kehvade suuskadega kaugele ei sõida .",no,"The phrase 'hooldeprojekti' refers to a project or initiative, not a geographic location, so it is not classified as a location.",no,"The phrase 'hooldeprojekti' was classified as 'no' because it does not indicate a location or answer the question 'where', but rather refers to a project related to care.",NaN,NaN,NaN,"[(hooldeprojekt, adt, S), (hooldeprojekt, sg g, S), (hooldeprojekt, sg p, S)]"
73,2891090,4635435,3,saama,NaN,adit,Risbiter,Risbiteri,[ Saab Risbiteri mõõga kätte .,no,The phrase 'Risbiteri' was classified as 'no' because it appears to refer to a specific object (possibly a sword) rather than a location.,no,The phrase 'Risbiteri' was not classified as an adverbial of place because it refers to a proper noun (likely a name) rather than a location or spatial context.,NaN,NaN,NaN,"[(Risbiter, adt, H), (Risbiter, sg g, H), (Risbiter, sg p, H), (Risbiteri, sg g, H), (Risbiteri, sg n, H)]"
88,14293818,22692055,3,laskma,NaN,adit,Narusk,Naruski,Mae laskis Naruski üheksasekundilisse eduseisu ja alustas siis karmi tagaajamist .,no,'Naruski' is not classified as a location because it appears to refer to a person's name rather than a geographical location.,no,"The phrase 'Naruski' was classified as 'no' because it refers to a person and not a place, so it does not indicate an adverbial of place.",NaN,NaN,LOC,"[(Narusk, adt, H), (Narusk, sg g, H), (Narusk, sg p, H), (Naruski, sg g, H), (Naruski, sg n, H)]"
98,2066247,3296559,23,pagema,NaN,adit,peit,peitu,"Valdo Lips lõi tempo alla , Ivo Saksakulm ja Chris Moore kontrollisid lauavõitlust , Indrek Varblane ja Andrus Nagel pagesid kaitse haardest peitu ja tulistasid kaugvisetega .",no,"The phrase 'peitu' was classified as 'no' because it describes a state or action of hiding, not a location.",no,The phrase 'peitu' is not classified as an adverbial of place because it indicates a movement to hide rather than specifying a precise location.,NaN,NaN,NaN,"[(peit, adt, S), (peituma, o, V), (peit, sg p, S)]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9861,8665558,13903441,5,ehitama,NaN,in,Ikaru,Ikarus,Viimasel kahel aastal on Ikarus ehitanud umbes 3000 bussi .,no,"The phrase 'Ikarus' refers to a company name, not a physical location, which is why it is not classified as a location.",no,"The phrase 'Ikarus' is not an adverbial of place because it refers to the subject performing the action, not a location.",NaN,NaN,LOC,"[(Ikaru, sg in, H), (Ikarus, sg n, H)]"
9930,16723199,25730032,15,jälgima,NaN,in,kukal,kuklas,"Esimesena võetakse üles stseen Kaks inimest sooserval , kus eit ja taat , pead kuklas , üle taeva vihisevat heledat kera jälgivad .",no,"The phrase 'kuklas' refers to the back of the head, which is a body part and not a location, so it was classified as 'no'.",no,"The phrase 'kuklas' refers to the position of the head (back of the head) rather than indicating a location in space, so it was classified as 'no'.",NaN,NaN,NaN,"[(kuklas, , A), (kuklas, , D), (kukal, sg in, S)]"
9944,3338090,5368550,1,liikuma,NaN,adit,lennuplaan,Lennuplaani,"Lennuplaani järgselt Kaliningradi suunas liikunud lennuk sisenes Eesti õhuruumi Vaindloo saare piirkonnas ühe meremiili sügavuselt , viibides Eesti õhuruumis 

## kas on omadussõna või viisimäärus (adj A või määrsõna adv D)

In [281]:
df_no = df1[df1["classification2"]=="no"]

In [276]:
def get_analysis4(ex):
    txt = Text(ex["sentence"])
    txt.tag_layer("words")
    txt.tag_layer("sentences")
    morf_tagger.tag(txt)
    txt.tag_layer('morph_extended')
    stanza_tagger.tag( txt )
    
    head_loc = int(ex["head_loc"])-1
    head_analysis = txt.morph_analysis[head_loc]
    
    roots = list(head_analysis.root)
    lemmas = list(head_analysis.lemma)
    forms = list(head_analysis.form)
    poss = list(head_analysis.partofspeech)

    variants = []
    for r, f, pos in zip(lemmas, forms, poss):
        if pos in ["A", "D"]:
            variants.append((r, f, pos))

    return txt, txt.morph_analysis, head_analysis, variants

In [277]:
ad_data = []

for i in tqdm(range(len(df_no))):
    #ex = df1[df1["sentence"].str.contains("Raudteelaste kraesse toimunud")]
    ex = df_no.iloc[i]
    #print(ex)
    #print(ex["sentence"])
    text, analysis, word_analysis, head_variants = get_analysis4(ex)
    if len(head_variants) > 0:
        #print(i, text, "\nhead_word:", ex["form"])
        #print(head_variants, "\n\n")
        ad_data.append(head_variants)
        #break
    else:
        ad_data.append("")

100%|███████████████████████████████████████| 1306/1306 [00:40<00:00, 32.43it/s]


In [282]:
df_no["adj_adv"] = ad_data

/tmp/ipykernel_32390/484933789.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_no["adj_adv"] = ad_data


In [294]:
df_no.to_csv(RESULT_DIR+ "n80_examples_large_v01_gpt_v02_10K_b10_v01_no_adj_adv.csv", encoding="utf-8", index=False, sep=",", quoting=csv.QUOTE_MINIMAL)

In [283]:
df_no

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,classification,explanation,classification2,explanation2,timex_tag,ekilex_tag,ner_tag,adj_adv
6,9344322,15008647,14,tervitama,NaN,in,ekstaas,ekstaasis,"Tahtsin lihtsalt lõpetada ja mõned punktid saada , ” tunnistas sakslane , keda ekstaasis publik autasustamispoodiumil ovatsioonide ja raketipaukudega tervitas .",no,"The word 'ekstaasis' translates to 'in ecstasy' and refers to an emotional state, not a physical location.",no,"The phrase 'ekstaasis' refers to an emotional state and does not indicate a physical location, so it is not adverbial of place.",NaN,NaN,NaN,
8,11102573,17784543,6,ravima,NaN,in,seis,seisus,"Kaks kuud ravis perenaine armetus seisus koera , kaotamata siiski lootust .",no,The phrase 'seisus' refers to 'condition' or 'state' and does not signify a location.,no,"The phrase 'seisus' refers to a condition or state rather than a location, so it is not adverbial of place.",NaN,NaN,NaN,
14,16252383,25172405,39,minema,ära,el,koosseis,koosseisust,"Kolm näidet , mis ma oskan öelda nende kohta , kes on ise lahkunud ( peale selle on lahkujaid ka seoses struktuuri ümberkorraldamisega , ministeeriumist on ära viidud näiteks haldusbüroo ja tehtud muid niisuguseid asju , mistõttu ministeeriumi koosseisust ära läinud isikuid on rohkem ) : näiteks Tallinna linn on saanud Haridusministeeriumist personalijuhi , kes oli töötanud ministeeriumis 13 aastat ja tahtis vaheldust ; üks daam läheb meil järgmisel nädalal Islandile mehele ; üks daam on läinud arvutiõpetajaks Rocca al Mare kooli .",no,The phrase 'koosseisust' pertains to organizational structure and not a geographic location.,no,"The phrase 'koosseisust' is not adverbial of place because it refers to membership or composition, rather than specifying a location.",NaN,NaN,NaN,
27,15500381,24205992,12,panema,NaN,adit,hooldeprojekt,hooldeprojekti,""" Vajaduse korral panevad Veerpalu ja loodetavasti õed Šmigunid isiklikku raha hooldeprojekti , sest kehvade suuskadega kaugele ei sõida .",no,"The phrase 'hooldeprojekti' refers to a project or initiative, not a geographic location, so it is not classified as a location.",no,"The phrase 'hooldeprojekti' was classified as 'no' because it does not indicate a location or answer the question 'where', but rather refers to a project related to care.",NaN,NaN,NaN,
32,18375902,27858061,22,ütlema,NaN,ill,eelnev,eelnevasse,Mind lihtsalt huvitas kuidas Sa vastaksid .. neile kahele küsimusele ja ka sellele kolmandale mille Teekäija esitas .. tõesti sekkumata üldse eelnevasse kus mida keegi ütles - tahtsin teada lihtsalt mida Sina arvad v ütled selle kohta !,no,"The word 'eelnevasse' refers to a preceding context or matter, not a physical or geographical location.",no,"The phrase 'eelnevasse' refers to something previously mentioned or an earlier situation, not indicating a specific place.",NaN,NaN,NaN,"[(eelnev, sg ill, A)]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9944,3338090,5368550,1,liikuma,NaN,adit,lennuplaan,Lennuplaani,"Lennuplaani järgselt Kaliningradi suunas liikunud lennuk sisenes Eesti õhuruumi Vaindloo saare piirkonnas ühe meremiili sügavuselt , viibides Eesti õhuruumis alla minuti .",no,The phrase 'Lennuplaani' refers to a flight schedule and not a physical location.,no,"The phrase 'Lennuplaani' refers to a flight schedule, which is a temporal or organizational concept rather than a physical location or place, so it is not adverbial of place.",NaN,NaN,NaN,
9946,1433193,2280193,3,käima,NaN,in,14,14-s,"110-st valimisringkonnast 14-s , nende seas ka kolmes pealinna Minski ringkonnas , ei käinud komisjoni väitel oma häält andmas üle poole valijatest ja seal korraldatakse valimiste teine voor .",no,"The phrase '14-s' refers to an ordinal number and not a physical location, hence it was classified as 'no'.",no,"The phrase '14-s' refers to the specific number of constituencies but does not describe a physical location or place,

In [284]:
df_no[df_no["adj_adv"]!=""] # 131

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,classification,explanation,classification2,explanation2,timex_tag,ekilex_tag,ner_tag,adj_adv
32,18375902,27858061,22,ütlema,NaN,ill,eelnev,eelnevasse,Mind lihtsalt huvitas kuidas Sa vastaksid .. neile kahele küsimusele ja ka sellele kolmandale mille Teekäija esitas .. tõesti sekkumata üldse eelnevasse kus mida keegi ütles - tahtsin teada lihtsalt mida Sina arvad v ütled selle kohta !,no,"The word 'eelnevasse' refers to a preceding context or matter, not a physical or geographical location.",no,"The phrase 'eelnevasse' refers to something previously mentioned or an earlier situation, not indicating a specific place.",NaN,NaN,NaN,"[(eelnev, sg ill, A)]"
202,10766169,17245553,8,ravima,NaN,in,kord,korras,"Konsultatsioonid ainult , muidugi ravib ta hobi korras omainimesi .",no,The phrase 'korras' refers to a state or condition rather than indicating a location.,no,'korras' was classified as 'no' because it does not indicate a place but rather a state or condition.,NaN,NaN,NaN,"[(korras, , A)]"
242,6889362,11073774,34,laskma,maha,in,kord,korras,"Võib-olla suri selle tüübi vanaema kusagil Siberis ja tema isa ei võetud ülikooli , sest ta oli rahvavaenlaste laps , aga tema vanaema vend kükitas kusagil Võrumaa metsas , kuni ta lasi haarangu korras maha tema oma sõber metsavend , kes osutus KGB nuhiks .",no,"The phrase 'korras' refers to 'in order' or 'arranged' and not a location, so it is classified as 'no'.",no,"The phrase 'korras' describes the manner in which the action occurred and does not indicate a specific location, so it is not an adverbial of place.",NaN,NaN,NaN,"[(korras, , A)]"
318,731611,1165533,19,pagema,NaN,el,jalg,jalust,"Sajandi viimastest Kihnu päevadest võtsid osa peamiselt Rootsis ja Kanadas elavad väliskihnlased , kes viiekümne aasta eest sõja jalust võõrsile pagesid .",no,"The term 'jalust' refers to a state or condition rather than a specific location, so it is classified as 'no'.",no,"The phrase 'jalust' refers to a cause or reason (escaping from danger), not a location, so it is not classified as an adverbial of place.",NaN,NaN,NaN,"[(jalust, , D)]"
344,16394694,25335947,8,ehitama,NaN,in,esi,ees-,"Vastuoksa , kreeklaste tagamehed ehitasid tõkke Eesti ees- ja tagaliini vahele .",no,The phrase 'ees-' is a prefix and does not denote a physical or geographical location.,no,The phrase 'ees-' is not adverbial of place; it is part of a compound word and does not denote a specific location.,NaN,NaN,NaN,"[(ees, , D)]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9630,17716571,27059695,12,mahtuma,ära,ill,minut,minutisse,"Pärast diskuteeriksime veidi pikemalt , siis mu vastus mahuks ära seitsmesse minutisse , mis kodukorra järgi on ette nähtud .",no,"The phrase 'minutisse' refers to a unit of time and not a specific geographical location, thus it is classified as 'no'.",no,The phrase 'minutisse' does not provide information about a location but rather specifies a time-related aspect.,NaN,NaN,NaN,"[(minutine, adt, A)]"
9743,310303,486932,7,säilima,NaN,in,avalik,avalikus,"Kus säilivad raamatud paremini , kas avalikus või eraraamatukogus ?",no,"'avalikus' was classified as not a location because it refers to a type of library, not a specific geographical place.",no,The phrase 'avalikus' was classified as 'no' because it describes a type of library (public) rather than specifying a location or place where the event occurs.,NaN,NaN,NaN,"[(avalik, sg in, A)]"
9753,9051914,14555010,18,sõitma,NaN,abl,ebakaine,ebakainelt,"Ebakainena on roolis olnud 37 protsenti juhtidest , rohkem kui üks protsent kõigist juhtidest sõidab aga pidevalt ebakainelt .",no,"The phrase 'ebakainelt' was classified as 'no' because it describes a condition or manner (intoxicated), not a location.",no,"The phrase 'ebakainelt' is classified as 'no' because it describes the manner of driving (intoxicated), not the location where the driving occurred.",NaN

In [46]:
ex = df1[df1["sentence"].str.contains("magas joobes mees Maardus")]
text, analysis, word_analysis, head_variants = get_analysis(ex)
print(text, "\n\nhead_word:", ex.iloc[0].form)
analysis

Text(text='25.jaanuaril kell 22.20 magas joobes mees Maardus Keemikute tänaval .') 

head_word: joobes


Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('25.', [{'normalized_text': '25.', 'lemma': '25.', 'root': '25.', 'root_tokens': ['25.'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'O'}]),
Span('jaanuaril', [{'normalized_text': 'jaanuaril', 'lemma': 'jaanuar', 'root': 'jaanuar', 'root_tokens': ['jaanuar'], 'ending': 'l', 'clitic': '', 'form': 'sg ad', 'partofspeech': 'S'}]),
Span('kell', [{'normalized_text': 'kell', 'lemma': 'kell', 'root': 'kell', 'root_tokens': ['kell'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('22.20', [{'normalized_text': '22.20', 'lemma': '22.20', 'root': '22.20', 'root_tokens': ['22.20'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'N'}]),
Span('magas', [{'normalized_text': 'magas', 'lemma': 'magama', 'root': 'maga', 'root_tokens': ['maga'], 'ending': 's', 'clitic': '', 'form': 's', 'partofspeech': 'V'}, {'normalized_text': 'magas', 'lemma': 'magas', 'root': 'magas', 'root_tokens': ['magas'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('joobes', [{'normalized_text': 'joobes', 'lemma': 'joove', 'root': 'joove', 'root_tokens': ['joove'], 'ending': 's', 'clitic': '', 'form': 'sg in', 'partofspeech': 'S'}]),
Span('mees', [{'normalized_text': 'mees', 'lemma': 'mees', 'root': 'mees', 'root_tokens': ['mees'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}, {'normalized_text': 'mees', 'lemma': 'mesi', 'root': 'mesi', 'root_tokens': ['mesi'], 'ending': 's', 'clitic': '', 'form': 'sg in', 'partofspeech': 'S'}]),
Span('Maardus', [{'normalized_text': 'Maardus', 'lemma': 'Maardu', 'root': 'Maardu', 'root_tokens': ['Maardu'], 'ending': 's', 'clitic': '', 'form': 'sg in', 'partofspeech': 'H'}]),
Span('Keemikute', [{'normalized_text': 'Keemikute', 'lemma': 'Keemik', 'root': 'Keemik', 'root_tokens': ['Keemik'], 'ending': 'te', 'clitic': '', 'form': 'pl g', 'partofspeech': 'H'}, {'normalized_text': 'Keemikute', 'lemma': 'Keemikute', 'root': 'Keemikute', 'root_tokens': ['Keemikute'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'H'}, {'normalized_text': 'Keemikute', 'lemma': 'Keemikute', 'root': 'Keemikute', 'root_tokens': ['Keemikute'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'H'}, {'normalized_text': 'Keemikute', 'lemma': 'keemik', 'root': 'keemik', 'root_tokens': ['keemik'], 'ending': 'te', 'clitic': '', 'form': 'pl g', 'partofspeech': 'S'}]),
Span('tänaval', [{'normalized_text': 'tänaval', 'lemma': 'tänav', 'root': 'tänav', 'root_tokens': ['tänav'], 'ending': 'l', 'clitic': '', 'form': 'sg ad', 'partofspeech': 'S'}]),
Span('.', [{'normalized_text': '.', 'lemma': '.', 'root': '.', 'root_tokens': ['.'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}])])

In [47]:
text.stanza_syntax

Layer(name='stanza_syntax', attributes=('id', 'lemma', 'upostag', 'xpostag', 'feats', 'head', 'deprel', 'deps', 'misc'), spans=SL[Span('25.', [{'id': 1, 'lemma': '25.', 'upostag': 'N', 'xpostag': 'N', 'feats': OrderedDict([('ord', 'ord'), ('<?>', '<?>'), ('roman', 'roman')]), 'head': 2, 'deprel': 'amod', 'deps': '_', 'misc': '_'}]),
Span('jaanuaril', [{'id': 2, 'lemma': 'jaanuar', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('ad', 'ad')]), 'head': 5, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('kell', [{'id': 3, 'lemma': 'kell', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('nom', 'nom')]), 'head': 5, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('22.20', [{'id': 4, 'lemma': '22.20', 'upostag': 'N', 'xpostag': 'N', 'feats': OrderedDict([('card', 'card'), ('<?>', '<?>'), ('digit', 'digit')]), 'head': 3, 'deprel': 'nummod', 'deps': '_', 'misc': '_'}]),
Span('magas', [{'id': 5, 'lemma': 'magama', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('main', 'main'), ('indic', 'indic'), ('impf', 'impf'), ('ps3', 'ps3'), ('sg', 'sg'), ('ps', 'ps'), ('af', 'af')]), 'head': 0, 'deprel': 'root', 'deps': '_', 'misc': '_'}]),
Span('joobes', [{'id': 6, 'lemma': 'joove', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('in', 'in')]), 'head': 5, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('mees', [{'id': 7, 'lemma': 'mees', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('nom', 'nom')]), 'head': 5, 'deprel': 'nsubj', 'deps': '_', 'misc': '_'}]),
Span('Maardus', [{'id': 8, 'lemma': 'Maardu', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('prop', 'prop'), ('sg', 'sg'), ('in', 'in')]), 'head': 5, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('Keemikute', [{'id': 9, 'lemma': 'Keemik', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('prop', 'prop'), ('pl', 'pl'), ('gen', 'gen')]), 'head': 10, 'deprel': 'nmod', 'deps': '_', 'misc': '_'}]),
Span('tänaval', [{'id': 10, 'lemma': 'tänav', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('com', 'com'), ('sg', 'sg'), ('ad', 'ad')]), 'head': 5, 'deprel': 'obl', 'deps': '_', 'misc': '_'}]),
Span('.', [{'id': 11, 'lemma': '.', 'upostag': 'Z', 'xpostag': 'Z', 'feats': OrderedDict(), 'head': 5, 'deprel': 'punct', 'deps': '_', 'misc': '_'}])])